<h1>Group by: split-apply-combine</h1>

<p>By “group by” we are referring to a process involving one or more of the following
steps:</p>

<ul>
<li><p><strong>Splitting</strong> the data into groups based on some criteria.</p></li>
<li><p><strong>Applying</strong> a function to each group independently.</p></li>
<li><p><strong>Combining</strong> the results into a data structure.</p></li>
</ul>

<p>Out of these, the split step is the most straightforward. In the apply step, we might wish to do one of the following:</p>

<ul>

<li><p><strong>Aggregation</strong>: compute a summary statistic (or statistics) for each group. Some examples:</p>
<blockquote>
<div><ul>
<li><p>Compute group sums or means.</p></li>
<li><p>Compute group sizes / counts.</p></li>
</ul>
</div></blockquote>
</li>

<li><p><strong>Transformation</strong>: perform some group-specific computations and return a
like-indexed object. Some examples:</p>
<blockquote>
<div><ul>
<li><p>Standardize data (zscore) within a group.</p></li>
<li><p>Filling NAs within groups with a value derived from each group.</p></li>
</ul>
</div></blockquote>
</li>

<li><p><strong>Filtration</strong>: discard some groups, according to a group-wise computation
that evaluates to True or False. Some examples:</p>
<blockquote>
<div><ul>
<li><p>Discard data that belong to groups with only a few members.</p></li>
<li><p>Filter out data based on the group sum or mean.</p></li>
</ul>
</div></blockquote>
</li>

</ul>

<p>Many of these operations are defined on GroupBy objects.
These operations are similar to those of the <a href="https://pandas.pydata.org/docs/user_guide/basics.html#basics-aggregate">aggregating API</a>, <a href="https://pandas.pydata.org/docs/user_guide/window.html#window-overview">window API</a>, and <a href="https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-aggregate">resample API</a>.</p>

<p>It is possible that a given operation does not fall into one of these categories or is some combination of them.
In such a case, it may be possible to compute the operation using GroupBy’s <code>apply</code> method.
This method will examine the results of the apply step and try to sensibly combine them into a single result if it doesn’t fit into either
of the above three categories.</p>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>An operation that is split into multiple steps using built-in GroupBy operations will be more efficient than using the <code>apply</code> method with a user-defined Python function.</p>
</div>

<p>The name GroupBy should be quite familiar to those who have used
a SQL-based tool (or <code class="docutils literal notranslate"><span class="pre">itertools</code>), in which you can write code like:</p>

<pre>
SELECT Column1, Column2, mean(Column3), sum(Column4)
FROM SomeTable
GROUP BY Column1, Column2
</pre>

<p>We aim to make operations like this natural and easy to express using
pandas. We’ll address each area of GroupBy functionality, then provide some non-trivial examples / use cases.</p>

<p>See the <a href="https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-grouping">cookbook</a> for some advanced strategies.</p>

# <h2>Splitting an object into groups</h2>

<p>The abstract definition of grouping is to provide a mapping of labels to group names. To create a GroupBy object (more on what the GroupBy object is later), you may do the following:</p>

In [2]:
import pandas as pd
import numpy as np

In [3]:
speeds = pd.DataFrame(
    [
        ("bird", "Falconiformes", 389.0),
        ("bird", "Psittaciformes", 24.0),
        ("mammal", "Carnivora", 80.2),
        ("mammal", "Primates", np.nan),
        ("mammal", "Carnivora", 58),
    ],
    index=["falcon", "parrot", "lion", "monkey", "leopard"],
    columns=("class", "order", "max_speed"),
)

In [4]:
speeds

,class,order,max_speed
falcon,bird,Falconiformes,389.0
parrot,bird,Psittaciformes,24.0
lion,mammal,Carnivora,80.2
monkey,mammal,Primates,NaN
leopard,mammal,Carnivora,58.0


In [5]:
grouped = speeds.groupby("class")

In [6]:
grouped = speeds.groupby(["class", "order"])

<p>The mapping can be specified many different ways:</p>
<ul>
<li><p>A Python function, to be called on each of the index labels.</p></li>
<li><p>A list or NumPy array of the same length as the index.</p></li>
<li><p>A dict or <code>Series</code>, providing a <code>label -&gt; group name</code> mapping.</p></li>
<li><p>For <code>DataFrame</code> objects, a string indicating either a column name or
an index level name to be used to group.</p></li>
<li><p>A list of any of the above things.</p></li>
</ul>

<p>Collectively we refer to the grouping objects as the <strong>keys</strong>. For example,
consider the following <code>DataFrame</code>:</p>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>A string passed to <code>groupby</code> may refer to either a column or an index level.
If a string matches both a column name and an index level name, a
<code>ValueError</code> will be raised.</p>

In [7]:
df = pd.DataFrame(
    {
        "A": ["foo", "bar", "foo", "bar", "foo", "bar", "foo", "foo"],
        "B": ["one", "one", "two", "three", "two", "two", "one", "three"],
        "C": np.random.randn(8),
        "D": np.random.randn(8),
    }
)

In [8]:
df

,A,B,C,D
0,foo,one,-0.117430,-0.056812
1,bar,one,0.717510,1.084787
2,foo,two,-2.084876,-0.492075
3,bar,three,0.016392,-1.324394
4,foo,two,0.314282,-0.695794
5,bar,two,1.903315,1.519822
6,foo,one,0.512739,-0.690980
7,foo,three,0.072287,0.208463


<p>On a DataFrame, we obtain a GroupBy object by calling <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html#pandas.DataFrame.groupby" title="pandas.DataFrame.groupby"><code>groupby()</code></a>.
This method returns a <code>pandas.api.typing.DataFrameGroupBy</code> instance.
We could naturally group by either the <code>A</code> or <code>B</code> columns, or both:</p>

In [9]:
grouped = df.groupby("A")

In [10]:
grouped = df.groupby("B")

In [11]:
grouped = df.groupby(["A", "B"])

<div class="admonition note">
<p class="admonition-title">Note</p>
<p><code>df.groupby('A')</code> is just syntactic sugar for <code>df.groupby(df['A'])</code>.</p>
</div>

<p>If we also have a MultiIndex on columns <code>A</code> and <code>B</code>, we can group by all the columns except the one we specify:</p>

In [12]:
 df2 = df.set_index(["A", "B"])

In [13]:
grouped = df2.groupby(level=df2.index.names.difference(["B"]))

In [14]:
grouped.sum()

,C,D
A,,
bar,2.637217,1.280215
foo,-1.302998,-1.727197


<p>The above GroupBy will split the DataFrame on its index (rows). To split by columns, first do
a transpose:</p>

In [15]:
def get_letter_type(letter):
    if letter.lower() in 'aeiou':
        return 'vowel'
    else:
        return 'consonant'

In [16]:
grouped = df.T.groupby(get_letter_type)

<p>pandas <a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html#pandas.Index"><code>Index</code></a> objects support duplicate values.
If a non-unique index is used as the group key in a groupby operation, all values for the same index value will be considered to be in one group and thus the output of aggregation functions will only contain unique index values:</p>

In [17]:
index = [1, 2, 3, 1, 2, 3]

In [18]:
s = pd.Series([1, 2, 3, 10, 20, 30], index=index)

In [19]:
s

,0
1,1
2,2
3,3
1,10
2,20
3,30


In [20]:
grouped = s.groupby(level=0)

In [21]:
grouped.first()

,0
1,1
2,2
3,3


In [22]:
grouped.last()

,0
1,10
2,20
3,30


In [23]:
grouped.sum()

,0
1,11
2,22
3,33


<p>Note that <strong>no splitting occurs</strong> until it’s needed. Creating the GroupBy object only verifies that you’ve passed a valid mapping.</p>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Many kinds of complicated data manipulations can be expressed in terms of GroupBy operations (though it can’t be guaranteed to be the most efficient implementation).
You can get quite creative with the label mapping functions.</p>
</div>

## <h3>GroupBy sorting</h3>

<p>By default the group keys are sorted during the <code>groupby</code> operation. You may however pass <code>sort=False</code> for potential speedups. With <code>sort=False</code> the order among group-keys follows the order of appearance of the keys in the original dataframe:</p>

In [24]:
df2 = pd.DataFrame({"X": ["B", "B", "A", "A"], "Y": [1, 2, 3, 4]})

In [25]:
df2.groupby(["X"]).sum()

,Y
X,
A,7
B,3


In [26]:
df2.groupby(["X"], sort=False).sum()

,Y
X,
B,3
A,7


<p>Note that <code>groupby</code> will preserve the order in which <em>observations</em> are sorted <em>within</em> each group.
For example, the groups created by <code>groupby()</code> below are in the order they appeared in the original <code>DataFrame</code>:</p>

In [27]:
df3 = pd.DataFrame({"X": ["A", "B", "A", "B"], "Y": [1, 4, 3, 2]})

In [28]:
df3.groupby("X").get_group("A")

,X,Y
0,A,1
2,A,3


In [29]:
df3.groupby(["X"]).get_group(("B",))

,X,Y
1,B,4
3,B,2


### <h4>GroupBy dropna</h4>

<p>By default <code>NA</code> values are excluded from group keys during the <code>groupby</code> operation.
However, in case you want to include <code>NA</code> values in group keys, you could pass <code>dropna=False</code> to achieve it.</p>

In [1]:
df_list = [[1, 2, 3], [1, None, 4], [2, 1, 3], [1, 2, 2]]

In [30]:
df_dropna = pd.DataFrame(df_list, columns=["a", "b", "c"])

In [31]:
df_dropna

,a,b,c
0,1,2.0,3
1,1,NaN,4
2,2,1.0,3
3,1,2.0,2


In [33]:
# Default ``dropna`` is set to True, which will exclude NaNs in keys
df_dropna.groupby(by=["b"], dropna=True).sum()

,a,c
b,,
1.0,2,3
2.0,2,5


In [34]:
# In order to allow NaN in keys, set dropna to False
df_dropna.groupby(by=["b"], dropna=False).sum()

,a,c
b,,
1.0,2,3
2.0,2,5
NaN,1,4


<p>The default setting of <code>dropna</code> argument is <code>True</code> which means <code>NA</code> are not included in group keys.</p>

## <h3>GroupBy object attributes</h3>

<p>The <code>groups</code> attribute is a dictionary whose keys are the computed unique groups and corresponding values are the axis labels belonging to each group. In the above example we have:</p>

In [35]:
df.groupby("A").groups

{'bar': [1, 3, 5], 'foo': [0, 2, 4, 6, 7]}

In [36]:
df.T.groupby(get_letter_type).groups

{'consonant': ['B', 'C', 'D'], 'vowel': ['A']}

<p>Calling the standard Python <code>len</code> function on the GroupBy object returns the number of groups, which is the same as the length of the <code>groups</code> dictionary:</p>

In [37]:
grouped = df.groupby(["A", "B"])

In [38]:
grouped.groups

{('bar', 'one'): [1], ('bar', 'three'): [3], ('bar', 'two'): [5], ('foo', 'one'): [0, 6], ('foo', 'three'): [7], ('foo', 'two'): [2, 4]}

In [39]:
len(grouped)

6

<p><code>GroupBy</code> will tab complete column names, GroupBy operations, and other attributes:</p>

In [40]:
n = 10

In [41]:
weight = np.random.normal(166, 20, size=n)

In [42]:
height = np.random.normal(60, 10, size=n)

In [43]:
time = pd.date_range("1/1/2000", periods=n)

In [44]:
gender = np.random.choice(["male", "female"], size=n)

In [45]:
df = pd.DataFrame(
    {"height": height, "weight": weight, "gender": gender}, index=time
)

In [46]:
df

,height,weight,gender
2000-01-01,58.673798,185.147707,male
2000-01-02,60.330306,171.722928,male
2000-01-03,68.358808,179.114289,male
2000-01-04,79.065763,186.391964,male
2000-01-05,59.929085,178.956307,male
2000-01-06,53.343484,181.812035,male
2000-01-07,56.900384,168.397189,male
2000-01-08,61.341789,132.421338,female
2000-01-09,69.765555,155.717252,female
2000-01-10,63.595336,137.122733,female


In [47]:
gb = df.groupby("gender")

<pre>
In [46]: gb.<TAB>  # noqa: E225, E999
gb.agg        gb.boxplot    gb.cummin     gb.describe   gb.filter     gb.get_group  gb.height     gb.last       gb.median     gb.ngroups    gb.plot       gb.rank       gb.std        gb.transform
gb.aggregate  gb.count      gb.cumprod    gb.dtype      gb.first      gb.groups     gb.hist       gb.max        gb.min        gb.nth        gb.prod       gb.resample   gb.sum        gb.var
gb.apply      gb.cummax     gb.cumsum     gb.fillna     gb.gender     gb.head       gb.indices    gb.mean       gb.name       gb.ohlc       gb.quantile   gb.size       gb.tail       gb.weight
</pre>

## <h3>GroupBy with MultiIndex</h3>

<p>With <a href="https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-hierarchical">hierarchically-indexed data</a>, it’s quite
natural to group by one of the levels of the hierarchy.</p>

<p>Let’s create a Series with a two-level <code>MultiIndex</code>.</p>

In [50]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

In [51]:
index = pd.MultiIndex.from_arrays(arrays, names=["first", "second"])

In [52]:
s = pd.Series(np.random.randn(8), index=index)

In [53]:
s

first  second
bar    one       1.059391
       two      -1.238853
baz    one       0.179113
       two       0.650465
foo    one       0.845075
       two       2.014760
qux    one      -0.119046
       two      -0.423211
dtype: float64

<p>We can then group by one of the levels in <code>s</code>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell16"><span></span><span class="gp">In [51]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">s</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="mi">0</span><span class="p">)</span>

<span class="gp">In [52]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[52]: </span>
<span class="go">first</span>
<span class="go">bar   -0.962232</span>
<span class="go">baz    1.237723</span>
<span class="go">foo    0.785980</span>
<span class="go">qux    1.911055</span>
<span class="go">dtype: float64</span>
</pre>
</div>
</div>

<p>If the MultiIndex has names specified, these can be passed instead of the level number:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell17"><span></span><span class="gp">In [53]: </span><span class="n">s</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="s2">"second"</span><span class="p">)</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[53]: </span>
<span class="go">second</span>
<span class="go">one    0.980950</span>
<span class="go">two    1.991575</span>
<span class="go">dtype: float64</span>
</pre>
</div>
</div>

<p>Grouping with multiple levels is supported.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell18"><span></span><span class="gp">In [54]: </span><span class="n">arrays</span> <span class="o">=</span> <span class="p">[</span>
<span class="gp"></span>    <span class="p">[</span><span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"baz"</span><span class="p">,</span> <span class="s2">"baz"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"qux"</span><span class="p">,</span> <span class="s2">"qux"</span><span class="p">],</span>
<span class="gp"></span>    <span class="p">[</span><span class="s2">"doo"</span><span class="p">,</span> <span class="s2">"doo"</span><span class="p">,</span> <span class="s2">"bee"</span><span class="p">,</span> <span class="s2">"bee"</span><span class="p">,</span> <span class="s2">"bop"</span><span class="p">,</span> <span class="s2">"bop"</span><span class="p">,</span> <span class="s2">"bop"</span><span class="p">,</span> <span class="s2">"bop"</span><span class="p">],</span>
<span class="gp"></span>    <span class="p">[</span><span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">],</span>
<span class="gp"></span><span class="p">]</span>
<span class="gp"></span>

<span class="gp">In [55]: </span><span class="n">index</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">MultiIndex</span><span class="o">.</span><span class="n">from_arrays</span><span class="p">(</span><span class="n">arrays</span><span class="p">,</span> <span class="n">names</span><span class="o">=</span><span class="p">[</span><span class="s2">"first"</span><span class="p">,</span> <span class="s2">"second"</span><span class="p">,</span> <span class="s2">"third"</span><span class="p">])</span>

<span class="gp">In [56]: </span><span class="n">s</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">8</span><span class="p">),</span> <span class="n">index</span><span class="o">=</span><span class="n">index</span><span class="p">)</span>

<span class="gp">In [57]: </span><span class="n">s</span>
<span class="gh">Out[57]: </span>
<span class="go">first  second  third</span>
<span class="go">bar    doo     one     -1.131345</span>
<span class="go">               two     -0.089329</span>
<span class="go">baz    bee     one      0.337863</span>
<span class="go">               two     -0.945867</span>
<span class="go">foo    bop     one     -0.932132</span>
<span class="go">               two      1.956030</span>
<span class="go">qux    bop     one      0.017587</span>
<span class="go">               two     -0.016692</span>
<span class="go">dtype: float64</span>

<span class="gp">In [58]: </span><span class="n">s</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="p">[</span><span class="s2">"first"</span><span class="p">,</span> <span class="s2">"second"</span><span class="p">])</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[58]: </span>
<span class="go">first  second</span>
<span class="go">bar    doo      -1.220674</span>
<span class="go">baz    bee      -0.608004</span>
<span class="go">foo    bop       1.023898</span>
<span class="go">qux    bop       0.000895</span>
<span class="go">dtype: float64</span>
</pre>
</div>
</div>

<p>Index level names may be supplied as keys.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell19"><span></span><span class="gp">In [59]: </span><span class="n">s</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"first"</span><span class="p">,</span> <span class="s2">"second"</span><span class="p">])</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[59]: </span>
<span class="go">first  second</span>
<span class="go">bar    doo      -1.220674</span>
<span class="go">baz    bee      -0.608004</span>
<span class="go">foo    bop       1.023898</span>
<span class="go">qux    bop       0.000895</span>
<span class="go">dtype: float64</span>
</pre>
</div>
</div>

<p>More on the <code>sum</code> function and aggregation later.</p>

## <h3>Grouping DataFrame with Index levels and columns</h3>

<p>A DataFrame may be grouped by a combination of columns and index levels. You can specify both column and index names, or use a <a href="https://pandas.pydata.org/docs/reference/api/pandas.Grouper.html#pandas.Grouper" title="pandas.Grouper"><code>Grouper</code></a>.</p>

<p>Let’s first create a DataFrame with a MultiIndex:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell20"><span></span><span class="gp">In [60]: </span><span class="n">arrays</span> <span class="o">=</span> <span class="p">[</span>
<span class="gp"></span>    <span class="p">[</span><span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"baz"</span><span class="p">,</span> <span class="s2">"baz"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"qux"</span><span class="p">,</span> <span class="s2">"qux"</span><span class="p">],</span>
<span class="gp"></span>    <span class="p">[</span><span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">],</span>
<span class="gp"></span><span class="p">]</span>
<span class="gp"></span>

<span class="gp">In [61]: </span><span class="n">index</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">MultiIndex</span><span class="o">.</span><span class="n">from_arrays</span><span class="p">(</span><span class="n">arrays</span><span class="p">,</span> <span class="n">names</span><span class="o">=</span><span class="p">[</span><span class="s2">"first"</span><span class="p">,</span> <span class="s2">"second"</span><span class="p">])</span>

<span class="gp">In [62]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">({</span><span class="s2">"A"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="mi">3</span><span class="p">],</span> <span class="s2">"B"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">arange</span><span class="p">(</span><span class="mi">8</span><span class="p">)},</span> <span class="n">index</span><span class="o">=</span><span class="n">index</span><span class="p">)</span>

<span class="gp">In [63]: </span><span class="n">df</span>
<span class="gh">Out[63]: </span>
<span class="go">              A  B</span>
<span class="go">first second      </span>
<span class="go">bar   one     1  0</span>
<span class="go">      two     1  1</span>
<span class="go">baz   one     1  2</span>
<span class="go">      two     1  3</span>
<span class="go">foo   one     2  4</span>
<span class="go">      two     2  5</span>
<span class="go">qux   one     3  6</span>
<span class="go">      two     3  7</span>
</pre>
</div>
</div>
<p>Then we group <code>df</span></code> by the <code>second</span></code> index level and the <code>A</span></code> column.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell21"><span></span><span class="gp">In [64]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">pd</span><span class="o">.</span><span class="n">Grouper</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="mi">1</span><span class="p">),</span> <span class="s2">"A"</span><span class="p">])</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[64]: </span>
<span class="go">          B</span>
<span class="go">second A   </span>
<span class="go">one    1  2</span>
<span class="go">       2  4</span>
<span class="go">       3  6</span>
<span class="go">two    1  4</span>
<span class="go">       2  5</span>
<span class="go">       3  7</span>
</pre>
</div>
</div>

<p>Index levels may also be specified by name.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell22"><span></span><span class="gp">In [65]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">pd</span><span class="o">.</span><span class="n">Grouper</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="s2">"second"</span><span class="p">),</span> <span class="s2">"A"</span><span class="p">])</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[65]: </span>
<span class="go">          B</span>
<span class="go">second A   </span>
<span class="go">one    1  2</span>
<span class="go">       2  4</span>
<span class="go">       3  6</span>
<span class="go">two    1  4</span>
<span class="go">       2  5</span>
<span class="go">       3  7</span>
</pre>
</div>
</div>

<p>Index level names may be specified as keys directly to <code>groupby</code>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell23"><span></span><span class="gp">In [66]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"second"</span><span class="p">,</span> <span class="s2">"A"</span><span class="p">])</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[66]: </span>
<span class="go">          B</span>
<span class="go">second A   </span>
<span class="go">one    1  2</span>
<span class="go">       2  4</span>
<span class="go">       3  6</span>
<span class="go">two    1  4</span>
<span class="go">       2  5</span>
<span class="go">       3  7</span>
</pre>
</div>
</div>

## <h3>DataFrame column selection in GroupBy</h3>

<p>Once you have created the GroupBy object from a DataFrame, you might want to do something different for each of the columns.
Thus, by using <code>[]</span></code> on the GroupBy object in a similar way as the one used to get a column from a DataFrame, you can do:</p>


<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell24"><span></span><span class="gp">In [67]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp"></span>    <span class="p">{</span>
<span class="gp"></span>        <span class="s2">"A"</span><span class="p">:</span> <span class="p">[</span><span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">],</span>
<span class="gp"></span>        <span class="s2">"B"</span><span class="p">:</span> <span class="p">[</span><span class="s2">"one"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"three"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">,</span> <span class="s2">"three"</span><span class="p">],</span>
<span class="gp"></span>        <span class="s2">"C"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">8</span><span class="p">),</span>
<span class="gp"></span>        <span class="s2">"D"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">8</span><span class="p">),</span>
<span class="gp"></span>    <span class="p">}</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [68]: </span><span class="n">df</span>
<span class="gh">Out[68]: </span>
<span class="go">     A      B         C         D</span>
<span class="go">0  foo    one -0.575247  1.346061</span>
<span class="go">1  bar    one  0.254161  1.511763</span>
<span class="go">2  foo    two -1.143704  1.627081</span>
<span class="go">3  bar  three  0.215897 -0.990582</span>
<span class="go">4  foo    two  1.193555 -0.441652</span>
<span class="go">5  bar    two -0.077118  1.211526</span>
<span class="go">6  foo    one -0.408530  0.268520</span>
<span class="go">7  foo  three -0.862495  0.024580</span>

<span class="gp">In [69]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">])</span>

<span class="gp">In [70]: </span><span class="n">grouped_C</span> <span class="o">=</span> <span class="n">grouped</span><span class="p">[</span><span class="s2">"C"</span><span class="p">]</span>

<span class="gp">In [71]: </span><span class="n">grouped_D</span> <span class="o">=</span> <span class="n">grouped</span><span class="p">[</span><span class="s2">"D"</span><span class="p">]</span>
</pre>
</div>
</div>
<p>This is mainly syntactic sugar for the alternative, which is much more verbose:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell25"><span></span><span class="gp">In [72]: </span><span class="n">df</span><span class="p">[</span><span class="s2">"C"</span><span class="p">]</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">df</span><span class="p">[</span><span class="s2">"A"</span><span class="p">])</span>
<span class="gh">Out[72]: </span><span class="go">&lt;pandas.core.groupby.generic.SeriesGroupBy object at 0x7f10570765f0&gt;</span>
</pre>
</div>
</div>
<p>Additionally, this method avoids recomputing the internal grouping information
derived from the passed key.</p>
<p>You can also include the grouping columns if you want to operate on them.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell26"><span></span><span class="gp">In [73]: </span><span class="n">grouped</span><span class="p">[[</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">]]</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[73]: </span>
<span class="go">                   A                  B</span>
<span class="go">A                                      </span>
<span class="go">bar        barbarbar        onethreetwo</span>
<span class="go">foo  foofoofoofoofoo  onetwotwoonethree</span>
</pre>
</div>
</div>
</section>
</section>
<section id="iterating-through-groups">
<span id="groupby-iterating-label"></span><h2>Iterating through groups<a class="headerlink" href="#iterating-through-groups" title="Link to this heading">#</a></h2>
<p>With the GroupBy object in hand, iterating through the grouped data is very
natural and functions similarly to <a class="reference external" href="https://docs.python.org/3/library/itertools.html#itertools.groupby" title="(in Python v3.13)"><code class="xref py py-func docutils literal notranslate">itertools.groupby()</span></code></a>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell27"><span></span><span class="gp">In [74]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s1">'A'</span><span class="p">)</span>

<span class="gp">In [75]: </span><span class="k">for</span> <span class="n">name</span><span class="p">,</span> <span class="n">group</span> <span class="ow">in</span> <span class="n">grouped</span><span class="p">:</span>
<span class="gp"></span>    <span class="nb">print</span><span class="p">(</span><span class="n">name</span><span class="p">)</span>
<span class="gp"></span>    <span class="nb">print</span><span class="p">(</span><span class="n">group</span><span class="p">)</span>
<span class="gp"></span>
<span class="go">bar</span>
<span class="go">     A      B         C         D</span>
<span class="go">1  bar    one  0.254161  1.511763</span>
<span class="go">3  bar  three  0.215897 -0.990582</span>
<span class="go">5  bar    two -0.077118  1.211526</span>
<span class="go">foo</span>
<span class="go">     A      B         C         D</span>
<span class="go">0  foo    one -0.575247  1.346061</span>
<span class="go">2  foo    two -1.143704  1.627081</span>
<span class="go">4  foo    two  1.193555 -0.441652</span>
<span class="go">6  foo    one -0.408530  0.268520</span>
<span class="go">7  foo  three -0.862495  0.024580</span>
</pre>
</div>
</div>
<p>In the case of grouping by multiple keys, the group name will be a tuple:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell28"><span></span><span class="gp">In [76]: </span><span class="k">for</span> <span class="n">name</span><span class="p">,</span> <span class="n">group</span> <span class="ow">in</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s1">'A'</span><span class="p">,</span> <span class="s1">'B'</span><span class="p">]):</span>
<span class="gp"></span>    <span class="nb">print</span><span class="p">(</span><span class="n">name</span><span class="p">)</span>
<span class="gp"></span>    <span class="nb">print</span><span class="p">(</span><span class="n">group</span><span class="p">)</span>
<span class="gp"></span>
<span class="go">('bar', 'one')</span>
<span class="go">     A    B         C         D</span>
<span class="go">1  bar  one  0.254161  1.511763</span>
<span class="go">('bar', 'three')</span>
<span class="go">     A      B         C         D</span>
<span class="go">3  bar  three  0.215897 -0.990582</span>
<span class="go">('bar', 'two')</span>
<span class="go">     A    B         C         D</span>
<span class="go">5  bar  two -0.077118  1.211526</span>
<span class="go">('foo', 'one')</span>
<span class="go">     A    B         C         D</span>
<span class="go">0  foo  one -0.575247  1.346061</span>
<span class="go">6  foo  one -0.408530  0.268520</span>
<span class="go">('foo', 'three')</span>
<span class="go">     A      B         C        D</span>
<span class="go">7  foo  three -0.862495  0.02458</span>
<span class="go">('foo', 'two')</span>
<span class="go">     A    B         C         D</span>
<span class="go">2  foo  two -1.143704  1.627081</span>
<span class="go">4  foo  two  1.193555 -0.441652</span>
</pre>
</div>
</div>
<p>See <a href="timeseries.html#timeseries-iterating-label"><span class="std std-ref">Iterating through groups</span></a>.</p>
</section>
<section id="selecting-a-group">
<h2>Selecting a group<a class="headerlink" href="#selecting-a-group" title="Link to this heading">#</a></h2>
<p>A single group can be selected using
<a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.get_group.html#pandas.core.groupby.DataFrameGroupBy.get_group" title="pandas.core.groupby.DataFrameGroupBy.get_group"><code>DataFrameGroupBy.get_group()</span></code></a>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell29"><span></span><span class="gp">In [77]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">get_group</span><span class="p">(</span><span class="s2">"bar"</span><span class="p">)</span>
<span class="gh">Out[77]: </span>
<span class="go">     A      B         C         D</span>
<span class="go">1  bar    one  0.254161  1.511763</span>
<span class="go">3  bar  three  0.215897 -0.990582</span>
<span class="go">5  bar    two -0.077118  1.211526</span>
</pre>
</div>
</div>
<p>Or for an object grouped on multiple columns:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell30"><span></span><span class="gp">In [78]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span><span class="o">.</span><span class="n">get_group</span><span class="p">((</span><span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">))</span>
<span class="gh">Out[78]: </span>
<span class="go">     A    B         C         D</span>
<span class="go">1  bar  one  0.254161  1.511763</span>
</pre>
</div>
</div>
</section>
<section id="aggregation">
<span id="groupby-aggregate"></span><h2>Aggregation<a class="headerlink" href="#aggregation" title="Link to this heading">#</a></h2>
<p>An aggregation is a GroupBy operation that reduces the dimension of the grouping
object. The result of an aggregation is, or at least is treated as,
a scalar value for each column in a group. For example, producing the sum of each
column in a group of values.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell31"><span></span><span class="gp">In [79]: </span><span class="n">animals</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp"></span>    <span class="p">{</span>
<span class="gp"></span>        <span class="s2">"kind"</span><span class="p">:</span> <span class="p">[</span><span class="s2">"cat"</span><span class="p">,</span> <span class="s2">"dog"</span><span class="p">,</span> <span class="s2">"cat"</span><span class="p">,</span> <span class="s2">"dog"</span><span class="p">],</span>
<span class="gp"></span>        <span class="s2">"height"</span><span class="p">:</span> <span class="p">[</span><span class="mf">9.1</span><span class="p">,</span> <span class="mf">6.0</span><span class="p">,</span> <span class="mf">9.5</span><span class="p">,</span> <span class="mf">34.0</span><span class="p">],</span>
<span class="gp"></span>        <span class="s2">"weight"</span><span class="p">:</span> <span class="p">[</span><span class="mf">7.9</span><span class="p">,</span> <span class="mf">7.5</span><span class="p">,</span> <span class="mf">9.9</span><span class="p">,</span> <span class="mf">198.0</span><span class="p">],</span>
<span class="gp"></span>    <span class="p">}</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [80]: </span><span class="n">animals</span>
<span class="gh">Out[80]: </span>
<span class="go">  kind  height  weight</span>
<span class="go">0  cat     9.1     7.9</span>
<span class="go">1  dog     6.0     7.5</span>
<span class="go">2  cat     9.5     9.9</span>
<span class="go">3  dog    34.0   198.0</span>

<span class="gp">In [81]: </span><span class="n">animals</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"kind"</span><span class="p">)</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[81]: </span>
<span class="go">      height  weight</span>
<span class="go">kind                </span>
<span class="go">cat     18.6    17.8</span>
<span class="go">dog     40.0   205.5</span>
</pre>
</div>
</div>
<p>In the result, the keys of the groups appear in the index by default. They can be
instead included in the columns by passing <code>as_index=False</span></code>.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell32"><span></span><span class="gp">In [82]: </span><span class="n">animals</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"kind"</span><span class="p">,</span> <span class="n">as_index</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[82]: </span>
<span class="go">  kind  height  weight</span>
<span class="go">0  cat    18.6    17.8</span>
<span class="go">1  dog    40.0   205.5</span>
</pre>
</div>
</div>
<section id="built-in-aggregation-methods">
<span id="groupby-aggregate-builtin"></span><h3>Built-in aggregation methods<a class="headerlink" href="#built-in-aggregation-methods" title="Link to this heading">#</a></h3>
<p>Many common aggregations are built-in to GroupBy objects as methods. Of the methods
listed below, those with a <code>*</span></code> do <em>not</em> have an efficient, GroupBy-specific, implementation.</p>
<table class="table">
<colgroup>
<col style="width: 20.0%">
<col style="width: 80.0%">
</colgroup>
<thead>
<tr class="row-odd"><th class="head"><p>Method</p></th>
<th class="head"><p>Description</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.any.html#pandas.core.groupby.DataFrameGroupBy.any" title="pandas.core.groupby.DataFrameGroupBy.any"><code>any()</span></code></a></p></td>
<td><p>Compute whether any of the values in the groups are truthy</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.all.html#pandas.core.groupby.DataFrameGroupBy.all" title="pandas.core.groupby.DataFrameGroupBy.all"><code>all()</span></code></a></p></td>
<td><p>Compute whether all of the values in the groups are truthy</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.count.html#pandas.core.groupby.DataFrameGroupBy.count" title="pandas.core.groupby.DataFrameGroupBy.count"><code>count()</span></code></a></p></td>
<td><p>Compute the number of non-NA values in the groups</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.cov.html#pandas.core.groupby.DataFrameGroupBy.cov" title="pandas.core.groupby.DataFrameGroupBy.cov"><code>cov()</span></code></a> *</p></td>
<td><p>Compute the covariance of the groups</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.first.html#pandas.core.groupby.DataFrameGroupBy.first" title="pandas.core.groupby.DataFrameGroupBy.first"><code>first()</span></code></a></p></td>
<td><p>Compute the first occurring value in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.idxmax.html#pandas.core.groupby.DataFrameGroupBy.idxmax" title="pandas.core.groupby.DataFrameGroupBy.idxmax"><code>idxmax()</span></code></a></p></td>
<td><p>Compute the index of the maximum value in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.idxmin.html#pandas.core.groupby.DataFrameGroupBy.idxmin" title="pandas.core.groupby.DataFrameGroupBy.idxmin"><code>idxmin()</span></code></a></p></td>
<td><p>Compute the index of the minimum value in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.last.html#pandas.core.groupby.DataFrameGroupBy.last" title="pandas.core.groupby.DataFrameGroupBy.last"><code>last()</span></code></a></p></td>
<td><p>Compute the last occurring value in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.max.html#pandas.core.groupby.DataFrameGroupBy.max" title="pandas.core.groupby.DataFrameGroupBy.max"><code>max()</span></code></a></p></td>
<td><p>Compute the maximum value in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.mean.html#pandas.core.groupby.DataFrameGroupBy.mean" title="pandas.core.groupby.DataFrameGroupBy.mean"><code>mean()</span></code></a></p></td>
<td><p>Compute the mean of each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.median.html#pandas.core.groupby.DataFrameGroupBy.median" title="pandas.core.groupby.DataFrameGroupBy.median"><code>median()</span></code></a></p></td>
<td><p>Compute the median of each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.min.html#pandas.core.groupby.DataFrameGroupBy.min" title="pandas.core.groupby.DataFrameGroupBy.min"><code>min()</span></code></a></p></td>
<td><p>Compute the minimum value in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.nunique.html#pandas.core.groupby.DataFrameGroupBy.nunique" title="pandas.core.groupby.DataFrameGroupBy.nunique"><code>nunique()</span></code></a></p></td>
<td><p>Compute the number of unique values in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.prod.html#pandas.core.groupby.DataFrameGroupBy.prod" title="pandas.core.groupby.DataFrameGroupBy.prod"><code>prod()</span></code></a></p></td>
<td><p>Compute the product of the values in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.quantile.html#pandas.core.groupby.DataFrameGroupBy.quantile" title="pandas.core.groupby.DataFrameGroupBy.quantile"><code>quantile()</span></code></a></p></td>
<td><p>Compute a given quantile of the values in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.sem.html#pandas.core.groupby.DataFrameGroupBy.sem" title="pandas.core.groupby.DataFrameGroupBy.sem"><code>sem()</span></code></a></p></td>
<td><p>Compute the standard error of the mean of the values in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.size.html#pandas.core.groupby.DataFrameGroupBy.size" title="pandas.core.groupby.DataFrameGroupBy.size"><code>size()</span></code></a></p></td>
<td><p>Compute the number of values in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.skew.html#pandas.core.groupby.DataFrameGroupBy.skew" title="pandas.core.groupby.DataFrameGroupBy.skew"><code>skew()</span></code></a> *</p></td>
<td><p>Compute the skew of the values in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.std.html#pandas.core.groupby.DataFrameGroupBy.std" title="pandas.core.groupby.DataFrameGroupBy.std"><code>std()</span></code></a></p></td>
<td><p>Compute the standard deviation of the values in each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.sum.html#pandas.core.groupby.DataFrameGroupBy.sum" title="pandas.core.groupby.DataFrameGroupBy.sum"><code>sum()</span></code></a></p></td>
<td><p>Compute the sum of the values in each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.var.html#pandas.core.groupby.DataFrameGroupBy.var" title="pandas.core.groupby.DataFrameGroupBy.var"><code>var()</span></code></a></p></td>
<td><p>Compute the variance of the values in each group</p></td>
</tr>
</tbody>
</table>
<p>Some examples:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell33"><span></span><span class="gp">In [83]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)[[</span><span class="s2">"C"</span><span class="p">,</span> <span class="s2">"D"</span><span class="p">]]</span><span class="o">.</span><span class="n">max</span><span class="p">()</span>
<span class="gh">Out[83]: </span>
<span class="go">            C         D</span>
<span class="go">A                      </span>
<span class="go">bar  0.254161  1.511763</span>
<span class="go">foo  1.193555  1.627081</span>

<span class="gp">In [84]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>
<span class="gh">Out[84]: </span>
<span class="go">                  C         D</span>
<span class="go">A   B                        </span>
<span class="go">bar one    0.254161  1.511763</span>
<span class="go">    three  0.215897 -0.990582</span>
<span class="go">    two   -0.077118  1.211526</span>
<span class="go">foo one   -0.491888  0.807291</span>
<span class="go">    three -0.862495  0.024580</span>
<span class="go">    two    0.024925  0.592714</span>
</pre>
</div>
</div>
<p>Another aggregation example is to compute the size of each group.
This is included in GroupBy as the <code>size</span></code> method. It returns a Series whose
index consists of the group names and the values are the sizes of each group.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell34"><span></span><span class="gp">In [85]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span>

<span class="gp">In [86]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">size</span><span class="p">()</span>
<span class="gh">Out[86]: </span>
<span class="go">A    B    </span>
<span class="go">bar  one      1</span>
<span class="go">     three    1</span>
<span class="go">     two      1</span>
<span class="go">foo  one      2</span>
<span class="go">     three    1</span>
<span class="go">     two      2</span>
<span class="go">dtype: int64</span>
</pre>
</div>
</div>
<p>While the <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.describe.html#pandas.core.groupby.DataFrameGroupBy.describe" title="pandas.core.groupby.DataFrameGroupBy.describe"><code>DataFrameGroupBy.describe()</span></code></a> method is not itself a reducer, it
can be used to conveniently produce a collection of summary statistics about each of
the groups.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell35"><span></span><span class="gp">In [87]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">describe</span><span class="p">()</span>
<span class="gh">Out[87]: </span>
<span class="go">              C                      ...         D                    </span>
<span class="go">          count      mean       std  ...       50%       75%       max</span>
<span class="go">A   B                                ...                              </span>
<span class="go">bar one     1.0  0.254161       NaN  ...  1.511763  1.511763  1.511763</span>
<span class="go">    three   1.0  0.215897       NaN  ... -0.990582 -0.990582 -0.990582</span>
<span class="go">    two     1.0 -0.077118       NaN  ...  1.211526  1.211526  1.211526</span>
<span class="go">foo one     2.0 -0.491888  0.117887  ...  0.807291  1.076676  1.346061</span>
<span class="go">    three   1.0 -0.862495       NaN  ...  0.024580  0.024580  0.024580</span>
<span class="go">    two     2.0  0.024925  1.652692  ...  0.592714  1.109898  1.627081</span>

<span class="go">[6 rows x 16 columns]</span>
</pre>
</div>
</div>
<p>Another aggregation example is to compute the number of unique values of each group.
This is similar to the <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.value_counts.html#pandas.core.groupby.DataFrameGroupBy.value_counts" title="pandas.core.groupby.DataFrameGroupBy.value_counts"><code>DataFrameGroupBy.value_counts()</span></code></a> function, except that it only counts the
number of unique values.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell36"><span></span><span class="gp">In [88]: </span><span class="n">ll</span> <span class="o">=</span> <span class="p">[[</span><span class="s1">'foo'</span><span class="p">,</span> <span class="mi">1</span><span class="p">],</span> <span class="p">[</span><span class="s1">'foo'</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span> <span class="p">[</span><span class="s1">'foo'</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span> <span class="p">[</span><span class="s1">'bar'</span><span class="p">,</span> <span class="mi">1</span><span class="p">],</span> <span class="p">[</span><span class="s1">'bar'</span><span class="p">,</span> <span class="mi">1</span><span class="p">]]</span>

<span class="gp">In [89]: </span><span class="n">df4</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="n">ll</span><span class="p">,</span> <span class="n">columns</span><span class="o">=</span><span class="p">[</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span>

<span class="gp">In [90]: </span><span class="n">df4</span>
<span class="gh">Out[90]: </span>
<span class="go">     A  B</span>
<span class="go">0  foo  1</span>
<span class="go">1  foo  2</span>
<span class="go">2  foo  2</span>
<span class="go">3  bar  1</span>
<span class="go">4  bar  1</span>

<span class="gp">In [91]: </span><span class="n">df4</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)[</span><span class="s2">"B"</span><span class="p">]</span><span class="o">.</span><span class="n">nunique</span><span class="p">()</span>
<span class="gh">Out[91]: </span>
<span class="go">A</span>
<span class="go">bar    1</span>
<span class="go">foo    2</span>
<span class="go">Name: B, dtype: int64</span>
</pre>
</div>
</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Aggregation functions <strong>will not</strong> return the groups that you are aggregating over
as named <em>columns</em> when <code>as_index=True</span></code>, the default. The grouped columns will
be the <strong>indices</strong> of the returned object.</p>
<p>Passing <code>as_index=False</span></code> <strong>will</strong> return the groups that you are aggregating over as
named columns, regardless if they are named <strong>indices</strong> or <em>columns</em> in the inputs.</p>
</div>
</section>
<section id="the-aggregate-method">
<span id="groupby-aggregate-agg"></span><h3>The <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.aggregate.html#pandas.core.groupby.DataFrameGroupBy.aggregate" title="pandas.core.groupby.DataFrameGroupBy.aggregate"><code>aggregate()</span></code></a> method<a class="headerlink" href="#the-aggregate-method" title="Link to this heading">#</a></h3>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>The <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.aggregate.html#pandas.core.groupby.DataFrameGroupBy.aggregate" title="pandas.core.groupby.DataFrameGroupBy.aggregate"><code>aggregate()</span></code></a> method can accept many different types of
inputs. This section details using string aliases for various GroupBy methods; other
inputs are detailed in the sections below.</p>
</div>
<p>Any reduction method that pandas implements can be passed as a string to
<a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.aggregate.html#pandas.core.groupby.DataFrameGroupBy.aggregate" title="pandas.core.groupby.DataFrameGroupBy.aggregate"><code>aggregate()</span></code></a>. Users are encouraged to use the shorthand,
<code>agg</span></code>. It will operate as if the corresponding method was called.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell37"><span></span><span class="gp">In [92]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span>

<span class="gp">In [93]: </span><span class="n">grouped</span><span class="p">[[</span><span class="s2">"C"</span><span class="p">,</span> <span class="s2">"D"</span><span class="p">]]</span><span class="o">.</span><span class="n">aggregate</span><span class="p">(</span><span class="s2">"sum"</span><span class="p">)</span>
<span class="gh">Out[93]: </span>
<span class="go">            C         D</span>
<span class="go">A                      </span>
<span class="go">bar  0.392940  1.732707</span>
<span class="go">foo -1.796421  2.824590</span>

<span class="gp">In [94]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span>

<span class="gp">In [95]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span><span class="s2">"sum"</span><span class="p">)</span>
<span class="gh">Out[95]: </span>
<span class="go">                  C         D</span>
<span class="go">A   B                        </span>
<span class="go">bar one    0.254161  1.511763</span>
<span class="go">    three  0.215897 -0.990582</span>
<span class="go">    two   -0.077118  1.211526</span>
<span class="go">foo one   -0.983776  1.614581</span>
<span class="go">    three -0.862495  0.024580</span>
<span class="go">    two    0.049851  1.185429</span>
</pre>
</div>
</div>
<p>The result of the aggregation will have the group names as the
new index. In the case of multiple keys, the result is a
<a href="advanced.html#advanced-hierarchical"><span class="std std-ref">MultiIndex</span></a> by default. As mentioned above, this can be
changed by using the <code>as_index</span></code> option:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell38"><span></span><span class="gp">In [96]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">],</span> <span class="n">as_index</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>

<span class="gp">In [97]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span><span class="s2">"sum"</span><span class="p">)</span>
<span class="gh">Out[97]: </span>
<span class="go">     A      B         C         D</span>
<span class="go">0  bar    one  0.254161  1.511763</span>
<span class="go">1  bar  three  0.215897 -0.990582</span>
<span class="go">2  bar    two -0.077118  1.211526</span>
<span class="go">3  foo    one -0.983776  1.614581</span>
<span class="go">4  foo  three -0.862495  0.024580</span>
<span class="go">5  foo    two  0.049851  1.185429</span>

<span class="gp">In [98]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">,</span> <span class="n">as_index</span><span class="o">=</span><span class="kc">False</span><span class="p">)[[</span><span class="s2">"C"</span><span class="p">,</span> <span class="s2">"D"</span><span class="p">]]</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span><span class="s2">"sum"</span><span class="p">)</span>
<span class="gh">Out[98]: </span>
<span class="go">     A         C         D</span>
<span class="go">0  bar  0.392940  1.732707</span>
<span class="go">1  foo -1.796421  2.824590</span>
</pre>
</div>
</div>
<p>Note that you could use the <a href="../reference/api/pandas.DataFrame.reset_index.html#pandas.DataFrame.reset_index" title="pandas.DataFrame.reset_index"><code>DataFrame.reset_index()</span></code></a> DataFrame function to achieve
the same result as the column names are stored in the resulting <code>MultiIndex</span></code>, although
this will make an extra copy.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell39"><span></span><span class="gp">In [99]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span><span class="s2">"sum"</span><span class="p">)</span><span class="o">.</span><span class="n">reset_index</span><span class="p">()</span>
<span class="gh">Out[99]: </span>
<span class="go">     A      B         C         D</span>
<span class="go">0  bar    one  0.254161  1.511763</span>
<span class="go">1  bar  three  0.215897 -0.990582</span>
<span class="go">2  bar    two -0.077118  1.211526</span>
<span class="go">3  foo    one -0.983776  1.614581</span>
<span class="go">4  foo  three -0.862495  0.024580</span>
<span class="go">5  foo    two  0.049851  1.185429</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell39">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="aggregation-with-user-defined-functions">
<span id="groupby-aggregate-udf"></span><h3>Aggregation with User-Defined Functions<a class="headerlink" href="#aggregation-with-user-defined-functions" title="Link to this heading">#</a></h3>
<p>Users can also provide their own User-Defined Functions (UDFs) for custom aggregations.</p>
<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p>When aggregating with a UDF, the UDF should not mutate the
provided <code>Series</span></code>. See <a href="gotchas.html#gotchas-udf-mutation"><span class="std std-ref">Mutating with User Defined Function (UDF) methods</span></a> for more information.</p>
</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Aggregating with a UDF is often less performant than using
the pandas built-in methods on GroupBy. Consider breaking up a complex operation
into a chain of operations that utilize the built-in methods.</p>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell40"><span></span><span class="gp">In [100]: </span><span class="n">animals</span>
<span class="gh">Out[100]: </span>
<span class="go">  kind  height  weight</span>
<span class="go">0  cat     9.1     7.9</span>
<span class="go">1  dog     6.0     7.5</span>
<span class="go">2  cat     9.5     9.9</span>
<span class="go">3  dog    34.0   198.0</span>

<span class="gp">In [101]: </span><span class="n">animals</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"kind"</span><span class="p">)[[</span><span class="s2">"height"</span><span class="p">]]</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="nb">set</span><span class="p">(</span><span class="n">x</span><span class="p">))</span>
<span class="gh">Out[101]: </span>
<span class="go">           height</span>
<span class="go">kind             </span>
<span class="go">cat    {9.1, 9.5}</span>
<span class="go">dog   {34.0, 6.0}</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell40">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>The resulting dtype will reflect that of the aggregating function. If the results from different groups have
different dtypes, then a common dtype will be determined in the same way as <code>DataFrame</span></code> construction.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell41"><span></span><span class="gp">In [102]: </span><span class="n">animals</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"kind"</span><span class="p">)[[</span><span class="s2">"height"</span><span class="p">]]</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">astype</span><span class="p">(</span><span class="nb">int</span><span class="p">)</span><span class="o">.</span><span class="n">sum</span><span class="p">())</span>
<span class="gh">Out[102]: </span>
<span class="go">      height</span>
<span class="go">kind        </span>
<span class="go">cat       18</span>
<span class="go">dog       40</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell41">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="applying-multiple-functions-at-once">
<span id="groupby-aggregate-multifunc"></span><h3>Applying multiple functions at once<a class="headerlink" href="#applying-multiple-functions-at-once" title="Link to this heading">#</a></h3>
<p>On a grouped <code>Series</span></code>, you can pass a list or dict of functions to
<code>SeriesGroupBy.agg()</span></code>, outputting a DataFrame:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell42"><span></span><span class="gp">In [103]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span>

<span class="gp">In [104]: </span><span class="n">grouped</span><span class="p">[</span><span class="s2">"C"</span><span class="p">]</span><span class="o">.</span><span class="n">agg</span><span class="p">([</span><span class="s2">"sum"</span><span class="p">,</span> <span class="s2">"mean"</span><span class="p">,</span> <span class="s2">"std"</span><span class="p">])</span>
<span class="gh">Out[104]: </span>
<span class="go">          sum      mean       std</span>
<span class="go">A                                </span>
<span class="go">bar  0.392940  0.130980  0.181231</span>
<span class="go">foo -1.796421 -0.359284  0.912265</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell42">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>On a grouped <code>DataFrame</span></code>, you can pass a list of functions to
<code>DataFrameGroupBy.agg()</span></code> to aggregate each
column, which produces an aggregated result with a hierarchical column index:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell43"><span></span><span class="gp">In [105]: </span><span class="n">grouped</span><span class="p">[[</span><span class="s2">"C"</span><span class="p">,</span> <span class="s2">"D"</span><span class="p">]]</span><span class="o">.</span><span class="n">agg</span><span class="p">([</span><span class="s2">"sum"</span><span class="p">,</span> <span class="s2">"mean"</span><span class="p">,</span> <span class="s2">"std"</span><span class="p">])</span>
<span class="gh">Out[105]: </span>
<span class="go">            C                             D                    </span>
<span class="go">          sum      mean       std       sum      mean       std</span>
<span class="go">A                                                              </span>
<span class="go">bar  0.392940  0.130980  0.181231  1.732707  0.577569  1.366330</span>
<span class="go">foo -1.796421 -0.359284  0.912265  2.824590  0.564918  0.884785</span>
</pre>
</div>
</div>
<p>The resulting aggregations are named after the functions themselves. If you
need to rename, then you can add in a chained operation for a <code>Series</span></code> like this:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell44"><span></span><span class="gp">In [106]: </span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">grouped</span><span class="p">[</span><span class="s2">"C"</span><span class="p">]</span>
<span class="gp">   .....: </span>    <span class="o">.</span><span class="n">agg</span><span class="p">([</span><span class="s2">"sum"</span><span class="p">,</span> <span class="s2">"mean"</span><span class="p">,</span> <span class="s2">"std"</span><span class="p">])</span>
<span class="gp">   .....: </span>    <span class="o">.</span><span class="n">rename</span><span class="p">(</span><span class="n">columns</span><span class="o">=</span><span class="p">{</span><span class="s2">"sum"</span><span class="p">:</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"mean"</span><span class="p">:</span> <span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"std"</span><span class="p">:</span> <span class="s2">"baz"</span><span class="p">})</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>
<span class="gh">Out[106]: </span>
<span class="go">          foo       bar       baz</span>
<span class="go">A                                </span>
<span class="go">bar  0.392940  0.130980  0.181231</span>
<span class="go">foo -1.796421 -0.359284  0.912265</span>
</pre>
</div>
</div>
<p>For a grouped <code>DataFrame</span></code>, you can rename in a similar manner:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell45"><span></span><span class="gp">In [107]: </span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">grouped</span><span class="p">[[</span><span class="s2">"C"</span><span class="p">,</span> <span class="s2">"D"</span><span class="p">]]</span><span class="o">.</span><span class="n">agg</span><span class="p">([</span><span class="s2">"sum"</span><span class="p">,</span> <span class="s2">"mean"</span><span class="p">,</span> <span class="s2">"std"</span><span class="p">])</span><span class="o">.</span><span class="n">rename</span><span class="p">(</span>
<span class="gp">   .....: </span>        <span class="n">columns</span><span class="o">=</span><span class="p">{</span><span class="s2">"sum"</span><span class="p">:</span> <span class="s2">"foo"</span><span class="p">,</span> <span class="s2">"mean"</span><span class="p">:</span> <span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"std"</span><span class="p">:</span> <span class="s2">"baz"</span><span class="p">}</span>
<span class="gp">   .....: </span>    <span class="p">)</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>
<span class="gh">Out[107]: </span>
<span class="go">            C                             D                    </span>
<span class="go">          foo       bar       baz       foo       bar       baz</span>
<span class="go">A                                                              </span>
<span class="go">bar  0.392940  0.130980  0.181231  1.732707  0.577569  1.366330</span>
<span class="go">foo -1.796421 -0.359284  0.912265  2.824590  0.564918  0.884785</span>
</pre>
</div>
</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>In general, the output column names should be unique, but pandas will allow
you apply to the same function (or two functions with the same name) to the same
column.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell46"><span></span><span class="gp">In [108]: </span><span class="n">grouped</span><span class="p">[</span><span class="s2">"C"</span><span class="p">]</span><span class="o">.</span><span class="n">agg</span><span class="p">([</span><span class="s2">"sum"</span><span class="p">,</span> <span class="s2">"sum"</span><span class="p">])</span>
<span class="gh">Out[108]: </span>
<span class="go">          sum       sum</span>
<span class="go">A                      </span>
<span class="go">bar  0.392940  0.392940</span>
<span class="go">foo -1.796421 -1.796421</span>
</pre>
</div>
</div>
<p>pandas also allows you to provide multiple lambdas. In this case, pandas
will mangle the name of the (nameless) lambda functions, appending <code>_&lt;i&gt;</span></code>
to each subsequent lambda.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell47"><span></span><span class="gp">In [109]: </span><span class="n">grouped</span><span class="p">[</span><span class="s2">"C"</span><span class="p">]</span><span class="o">.</span><span class="n">agg</span><span class="p">([</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">max</span><span class="p">()</span> <span class="o">-</span> <span class="n">x</span><span class="o">.</span><span class="n">min</span><span class="p">(),</span> <span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">median</span><span class="p">()</span> <span class="o">-</span> <span class="n">x</span><span class="o">.</span><span class="n">mean</span><span class="p">()])</span>
<span class="gh">Out[109]: </span>
<span class="go">     &lt;lambda_0&gt;  &lt;lambda_1&gt;</span>
<span class="go">A                          </span>
<span class="go">bar    0.331279    0.084917</span>
<span class="go">foo    2.337259   -0.215962</span>
</pre>
</div>
</div>
</div>
</section>
<section id="named-aggregation">
<span id="groupby-aggregate-named"></span><h3>Named aggregation<a class="headerlink" href="#named-aggregation" title="Link to this heading">#</a></h3>
<p>To support column-specific aggregation <em>with control over the output column names</em>, pandas
accepts the special syntax in <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html#pandas.core.groupby.DataFrameGroupBy.agg" title="pandas.core.groupby.DataFrameGroupBy.agg"><code>DataFrameGroupBy.agg()</span></code></a> and <a href="../reference/api/pandas.core.groupby.SeriesGroupBy.agg.html#pandas.core.groupby.SeriesGroupBy.agg" title="pandas.core.groupby.SeriesGroupBy.agg"><code>SeriesGroupBy.agg()</span></code></a>, known as “named aggregation”, where</p>
<ul class="simple">
<li><p>The keywords are the <em>output</em> column names</p></li>
<li><p>The values are tuples whose first element is the column to select
and the second element is the aggregation to apply to that column. pandas
provides the <a href="../reference/api/pandas.NamedAgg.html#pandas.NamedAgg" title="pandas.NamedAgg"><code>NamedAgg</span></code></a> namedtuple with the fields <code>['column',</span> 'aggfunc']</span></code>
to make it clearer what the arguments are. As usual, the aggregation can
be a callable or a string alias.</p></li>
</ul>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell48"><span></span><span class="gp">In [110]: </span><span class="n">animals</span>
<span class="gh">Out[110]: </span>
<span class="go">  kind  height  weight</span>
<span class="go">0  cat     9.1     7.9</span>
<span class="go">1  dog     6.0     7.5</span>
<span class="go">2  cat     9.5     9.9</span>
<span class="go">3  dog    34.0   198.0</span>

<span class="gp">In [111]: </span><span class="n">animals</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"kind"</span><span class="p">)</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">min_height</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">NamedAgg</span><span class="p">(</span><span class="n">column</span><span class="o">=</span><span class="s2">"height"</span><span class="p">,</span> <span class="n">aggfunc</span><span class="o">=</span><span class="s2">"min"</span><span class="p">),</span>
<span class="gp">   .....: </span>    <span class="n">max_height</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">NamedAgg</span><span class="p">(</span><span class="n">column</span><span class="o">=</span><span class="s2">"height"</span><span class="p">,</span> <span class="n">aggfunc</span><span class="o">=</span><span class="s2">"max"</span><span class="p">),</span>
<span class="gp">   .....: </span>    <span class="n">average_weight</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">NamedAgg</span><span class="p">(</span><span class="n">column</span><span class="o">=</span><span class="s2">"weight"</span><span class="p">,</span> <span class="n">aggfunc</span><span class="o">=</span><span class="s2">"mean"</span><span class="p">),</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>
<span class="gh">Out[111]: </span>
<span class="go">      min_height  max_height  average_weight</span>
<span class="go">kind                                        </span>
<span class="go">cat          9.1         9.5            8.90</span>
<span class="go">dog          6.0        34.0          102.75</span>
</pre>
</div>
</div>
<p><a href="../reference/api/pandas.NamedAgg.html#pandas.NamedAgg" title="pandas.NamedAgg"><code>NamedAgg</span></code></a> is just a <code>namedtuple</span></code>. Plain tuples are allowed as well.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell49"><span></span><span class="gp">In [112]: </span><span class="n">animals</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"kind"</span><span class="p">)</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">min_height</span><span class="o">=</span><span class="p">(</span><span class="s2">"height"</span><span class="p">,</span> <span class="s2">"min"</span><span class="p">),</span>
<span class="gp">   .....: </span>    <span class="n">max_height</span><span class="o">=</span><span class="p">(</span><span class="s2">"height"</span><span class="p">,</span> <span class="s2">"max"</span><span class="p">),</span>
<span class="gp">   .....: </span>    <span class="n">average_weight</span><span class="o">=</span><span class="p">(</span><span class="s2">"weight"</span><span class="p">,</span> <span class="s2">"mean"</span><span class="p">),</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>
<span class="gh">Out[112]: </span>
<span class="go">      min_height  max_height  average_weight</span>
<span class="go">kind                                        </span>
<span class="go">cat          9.1         9.5            8.90</span>
<span class="go">dog          6.0        34.0          102.75</span>
</pre>
</div>
</div>
<p>If the column names you want are not valid Python keywords, construct a dictionary
and unpack the keyword arguments</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell50"><span></span><span class="gp">In [113]: </span><span class="n">animals</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"kind"</span><span class="p">)</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="o">**</span><span class="p">{</span>
<span class="gp">   .....: </span>        <span class="s2">"total weight"</span><span class="p">:</span> <span class="n">pd</span><span class="o">.</span><span class="n">NamedAgg</span><span class="p">(</span><span class="n">column</span><span class="o">=</span><span class="s2">"weight"</span><span class="p">,</span> <span class="n">aggfunc</span><span class="o">=</span><span class="s2">"sum"</span><span class="p">)</span>
<span class="gp">   .....: </span>    <span class="p">}</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>
<span class="gh">Out[113]: </span>
<span class="go">      total weight</span>
<span class="go">kind              </span>
<span class="go">cat           17.8</span>
<span class="go">dog          205.5</span>
</pre>
</div>
</div>
<p>When using named aggregation, additional keyword arguments are not passed through
to the aggregation functions; only pairs
of <code>(column,</span> aggfunc)</span></code> should be passed as <code class="docutils literal notranslate">**kwargs</span></code>. If your aggregation functions
require additional arguments, apply them partially with <code>functools.partial()</span></code>.</p>
<p>Named aggregation is also valid for Series groupby aggregations. In this case there’s
no column selection, so the values are just the functions.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell51"><span></span><span class="gp">In [114]: </span><span class="n">animals</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"kind"</span><span class="p">)</span><span class="o">.</span><span class="n">height</span><span class="o">.</span><span class="n">agg</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">min_height</span><span class="o">=</span><span class="s2">"min"</span><span class="p">,</span>
<span class="gp">   .....: </span>    <span class="n">max_height</span><span class="o">=</span><span class="s2">"max"</span><span class="p">,</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>
<span class="gh">Out[114]: </span>
<span class="go">      min_height  max_height</span>
<span class="go">kind                        </span>
<span class="go">cat          9.1         9.5</span>
<span class="go">dog          6.0        34.0</span>
</pre>
</div>
</div>
</section>
<section id="applying-different-functions-to-dataframe-columns">
<h3>Applying different functions to DataFrame columns<a class="headerlink" href="#applying-different-functions-to-dataframe-columns" title="Link to this heading">#</a></h3>
<p>By passing a dict to <code class="docutils literal notranslate">aggregate</span></code> you can apply a different aggregation to the
columns of a DataFrame:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell52"><span></span><span class="gp">In [115]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">agg</span><span class="p">({</span><span class="s2">"C"</span><span class="p">:</span> <span class="s2">"sum"</span><span class="p">,</span> <span class="s2">"D"</span><span class="p">:</span> <span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">std</span><span class="p">(</span><span class="n">x</span><span class="p">,</span> <span class="n">ddof</span><span class="o">=</span><span class="mi">1</span><span class="p">)})</span>
<span class="gh">Out[115]: </span>
<span class="go">            C         D</span>
<span class="go">A                      </span>
<span class="go">bar  0.392940  1.366330</span>
<span class="go">foo -1.796421  0.884785</span>
</pre>
</div>
</div>
<p>The function names can also be strings. In order for a string to be valid it
must be implemented on GroupBy:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell53"><span></span><span class="gp">In [116]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">agg</span><span class="p">({</span><span class="s2">"C"</span><span class="p">:</span> <span class="s2">"sum"</span><span class="p">,</span> <span class="s2">"D"</span><span class="p">:</span> <span class="s2">"std"</span><span class="p">})</span>
<span class="gh">Out[116]: </span>
<span class="go">            C         D</span>
<span class="go">A                      </span>
<span class="go">bar  0.392940  1.366330</span>
<span class="go">foo -1.796421  0.884785</span>
</pre>
</div>
</div>
</section>
</section>
<section id="transformation">
<span id="groupby-transform"></span><h2>Transformation<a class="headerlink" href="#transformation" title="Link to this heading">#</a></h2>
<p>A transformation is a GroupBy operation whose result is indexed the same
as the one being grouped. Common examples include <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.cumsum.html#pandas.core.groupby.DataFrameGroupBy.cumsum" title="pandas.core.groupby.DataFrameGroupBy.cumsum"><code>cumsum()</span></code></a> and
<a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.diff.html#pandas.core.groupby.DataFrameGroupBy.diff" title="pandas.core.groupby.DataFrameGroupBy.diff"><code>diff()</span></code></a>.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell54"><span></span><span class="gp">In [117]: </span><span class="n">speeds</span>
<span class="gh">Out[117]: </span>
<span class="go">          class           order  max_speed</span>
<span class="go">falcon     bird   Falconiformes      389.0</span>
<span class="go">parrot     bird  Psittaciformes       24.0</span>
<span class="go">lion     mammal       Carnivora       80.2</span>
<span class="go">monkey   mammal        Primates        NaN</span>
<span class="go">leopard  mammal       Carnivora       58.0</span>

<span class="gp">In [118]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">speeds</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"class"</span><span class="p">)[</span><span class="s2">"max_speed"</span><span class="p">]</span>

<span class="gp">In [119]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">cumsum</span><span class="p">()</span>
<span class="gh">Out[119]: </span>
<span class="go">falcon     389.0</span>
<span class="go">parrot     413.0</span>
<span class="go">lion        80.2</span>
<span class="go">monkey       NaN</span>
<span class="go">leopard    138.2</span>
<span class="go">Name: max_speed, dtype: float64</span>

<span class="gp">In [120]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">diff</span><span class="p">()</span>
<span class="gh">Out[120]: </span>
<span class="go">falcon       NaN</span>
<span class="go">parrot    -365.0</span>
<span class="go">lion         NaN</span>
<span class="go">monkey       NaN</span>
<span class="go">leopard      NaN</span>
<span class="go">Name: max_speed, dtype: float64</span>
</pre>
</div>
</div>
<p>Unlike aggregations, the groupings that are used to split
the original object are not included in the result.</p>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Since transformations do not include the groupings that are used to split the result,
the arguments <code class="docutils literal notranslate">as_index</span></code> and <code class="docutils literal notranslate">sort</span></code> in <a href="../reference/api/pandas.DataFrame.groupby.html#pandas.DataFrame.groupby" title="pandas.DataFrame.groupby"><code><span class="pre">DataFrame.groupby()</span></code></a> and
<a href="../reference/api/pandas.Series.groupby.html#pandas.Series.groupby" title="pandas.Series.groupby"><code><span class="pre">Series.groupby()</span></code></a> have no effect.</p>
</div>
<p>A common use of a transformation is to add the result back into the original DataFrame.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell55"><span></span><span class="gp">In [121]: </span><span class="n">result</span> <span class="o">=</span> <span class="n">speeds</span><span class="o">.</span><span class="n">copy</span><span class="p">()</span>

<span class="gp">In [122]: </span><span class="n">result</span><span class="p">[</span><span class="s2">"cumsum"</span><span class="p">]</span> <span class="o">=</span> <span class="n">grouped</span><span class="o">.</span><span class="n">cumsum</span><span class="p">()</span>

<span class="gp">In [123]: </span><span class="n">result</span><span class="p">[</span><span class="s2">"diff"</span><span class="p">]</span> <span class="o">=</span> <span class="n">grouped</span><span class="o">.</span><span class="n">diff</span><span class="p">()</span>

<span class="gp">In [124]: </span><span class="n">result</span>
<span class="gh">Out[124]: </span>
<span class="go">          class           order  max_speed  cumsum   diff</span>
<span class="go">falcon     bird   Falconiformes      389.0   389.0    NaN</span>
<span class="go">parrot     bird  Psittaciformes       24.0   413.0 -365.0</span>
<span class="go">lion     mammal       Carnivora       80.2    80.2    NaN</span>
<span class="go">monkey   mammal        Primates        NaN     NaN    NaN</span>
<span class="go">leopard  mammal       Carnivora       58.0   138.2    NaN</span>
</pre>
</div>
</div>
<section id="built-in-transformation-methods">
<h3>Built-in transformation methods<a class="headerlink" href="#built-in-transformation-methods" title="Link to this heading">#</a></h3>
<p>The following methods on GroupBy act as transformations.</p>
<table class="table">
<colgroup>
<col style="width: 20.0%">
<col style="width: 80.0%">
</colgroup>
<thead>
<tr class="row-odd"><th class="head"><p>Method</p></th>
<th class="head"><p>Description</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.bfill.html#pandas.core.groupby.DataFrameGroupBy.bfill" title="pandas.core.groupby.DataFrameGroupBy.bfill"><code><span class="pre">bfill()</span></code></a></p></td>
<td><p>Back fill NA values within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.cumcount.html#pandas.core.groupby.DataFrameGroupBy.cumcount" title="pandas.core.groupby.DataFrameGroupBy.cumcount"><code><span class="pre">cumcount()</span></code></a></p></td>
<td><p>Compute the cumulative count within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.cummax.html#pandas.core.groupby.DataFrameGroupBy.cummax" title="pandas.core.groupby.DataFrameGroupBy.cummax"><code><span class="pre">cummax()</span></code></a></p></td>
<td><p>Compute the cumulative max within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.cummin.html#pandas.core.groupby.DataFrameGroupBy.cummin" title="pandas.core.groupby.DataFrameGroupBy.cummin"><code><span class="pre">cummin()</span></code></a></p></td>
<td><p>Compute the cumulative min within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.cumprod.html#pandas.core.groupby.DataFrameGroupBy.cumprod" title="pandas.core.groupby.DataFrameGroupBy.cumprod"><code><span class="pre">cumprod()</span></code></a></p></td>
<td><p>Compute the cumulative product within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.cumsum.html#pandas.core.groupby.DataFrameGroupBy.cumsum" title="pandas.core.groupby.DataFrameGroupBy.cumsum"><code><span class="pre">cumsum()</span></code></a></p></td>
<td><p>Compute the cumulative sum within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.diff.html#pandas.core.groupby.DataFrameGroupBy.diff" title="pandas.core.groupby.DataFrameGroupBy.diff"><code><span class="pre">diff()</span></code></a></p></td>
<td><p>Compute the difference between adjacent values within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.ffill.html#pandas.core.groupby.DataFrameGroupBy.ffill" title="pandas.core.groupby.DataFrameGroupBy.ffill"><code><span class="pre">ffill()</span></code></a></p></td>
<td><p>Forward fill NA values within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.pct_change.html#pandas.core.groupby.DataFrameGroupBy.pct_change" title="pandas.core.groupby.DataFrameGroupBy.pct_change"><code><span class="pre">pct_change()</span></code></a></p></td>
<td><p>Compute the percent change between adjacent values within each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.rank.html#pandas.core.groupby.DataFrameGroupBy.rank" title="pandas.core.groupby.DataFrameGroupBy.rank"><code><span class="pre">rank()</span></code></a></p></td>
<td><p>Compute the rank of each value within each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.shift.html#pandas.core.groupby.DataFrameGroupBy.shift" title="pandas.core.groupby.DataFrameGroupBy.shift"><code><span class="pre">shift()</span></code></a></p></td>
<td><p>Shift values up or down within each group</p></td>
</tr>
</tbody>
</table>
<p>In addition, passing any built-in aggregation method as a string to
<a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html#pandas.core.groupby.DataFrameGroupBy.transform" title="pandas.core.groupby.DataFrameGroupBy.transform"><code><span class="pre">transform()</span></code></a> (see the next section) will broadcast the result
across the group, producing a transformed result. If the aggregation method has an efficient
implementation, this will be performant as well.</p>
</section>
<section id="the-transform-method">
<span id="groupby-transformation-transform"></span><h3>The <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html#pandas.core.groupby.DataFrameGroupBy.transform" title="pandas.core.groupby.DataFrameGroupBy.transform"><code><span class="pre">transform()</span></code></a> method<a class="headerlink" href="#the-transform-method" title="Link to this heading">#</a></h3>
<p>Similar to the <a href="#groupby-aggregate-agg"><span class="std std-ref">aggregation method</span></a>, the
<a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html#pandas.core.groupby.DataFrameGroupBy.transform" title="pandas.core.groupby.DataFrameGroupBy.transform"><code><span class="pre">transform()</span></code></a> method can accept string aliases to the built-in
transformation methods in the previous section. It can <em>also</em> accept string aliases to
the built-in aggregation methods. When an aggregation method is provided, the result
will be broadcast across the group.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell56"><span></span><span class="gp">In [125]: </span><span class="n">speeds</span>
<span class="gh">Out[125]: </span>
<span class="go">          class           order  max_speed</span>
<span class="go">falcon     bird   Falconiformes      389.0</span>
<span class="go">parrot     bird  Psittaciformes       24.0</span>
<span class="go">lion     mammal       Carnivora       80.2</span>
<span class="go">monkey   mammal        Primates        NaN</span>
<span class="go">leopard  mammal       Carnivora       58.0</span>

<span class="gp">In [126]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">speeds</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"class"</span><span class="p">)[[</span><span class="s2">"max_speed"</span><span class="p">]]</span>

<span class="gp">In [127]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="s2">"cumsum"</span><span class="p">)</span>
<span class="gh">Out[127]: </span>
<span class="go">         max_speed</span>
<span class="go">falcon       389.0</span>
<span class="go">parrot       413.0</span>
<span class="go">lion          80.2</span>
<span class="go">monkey         NaN</span>
<span class="go">leopard      138.2</span>

<span class="gp">In [128]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="s2">"sum"</span><span class="p">)</span>
<span class="gh">Out[128]: </span>
<span class="go">         max_speed</span>
<span class="go">falcon       413.0</span>
<span class="go">parrot       413.0</span>
<span class="go">lion         138.2</span>
<span class="go">monkey       138.2</span>
<span class="go">leopard      138.2</span>
</pre>
</div>
</div>
<p>In addition to string aliases, the <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html#pandas.core.groupby.DataFrameGroupBy.transform" title="pandas.core.groupby.DataFrameGroupBy.transform"><code><span class="pre">transform()</span></code></a> method can
also accept User-Defined Functions (UDFs). The UDF must:</p>
<ul class="simple">
<li><p>Return a result that is either the same size as the group chunk or
broadcastable to the size of the group chunk (e.g., a scalar,
<code class="docutils literal notranslate"><span class="pre">grouped.transform(lambda</span> <span class="pre">x:</span> <span class="pre">x.iloc[-1])</span></code>).</p></li>
<li><p>Operate column-by-column on the group chunk.  The transform is applied to
the first group chunk using chunk.apply.</p></li>
<li><p>Not perform in-place operations on the group chunk. Group chunks should
be treated as immutable, and changes to a group chunk may produce unexpected
results. See <a href="gotchas.html#gotchas-udf-mutation"><span class="std std-ref">Mutating with User Defined Function (UDF) methods</span></a> for more information.</p></li>
<li><p>(Optionally) operates on all columns of the entire group chunk at once. If this is
supported, a fast path is used starting from the <em>second</em> chunk.</p></li>
</ul>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Transforming by supplying <code class="docutils literal notranslate"><span class="pre">transform</span></code> with a UDF is
often less performant than using the built-in methods on GroupBy.
Consider breaking up a complex operation into a chain of operations that utilize
the built-in methods.</p>
<p>All of the examples in this section can be made more performant by calling
built-in methods instead of using UDFs.
See <a href="#groupby-efficient-transforms"><span class="std std-ref">below for examples</span></a>.</p>
</div>
<div class="versionchanged">
<p><span class="versionmodified changed">Changed in version 2.0.0: </span>When using <code class="docutils literal notranslate"><span class="pre">.transform</span></code> on a grouped DataFrame and the transformation function
returns a DataFrame, pandas now aligns the result’s index
with the input’s index. You can call <code class="docutils literal notranslate"><span class="pre">.to_numpy()</span></code> within the transformation
function to avoid alignment.</p>
</div>
<p>Similar to <a href="#groupby-aggregate-agg"><span class="std std-ref">The aggregate() method</span></a>, the resulting dtype will reflect that of the
transformation function. If the results from different groups have different dtypes, then
a common dtype will be determined in the same way as <code class="docutils literal notranslate"><span class="pre">DataFrame</span></code> construction.</p>
<p>Suppose we wish to standardize the data within each group:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell57"><span></span><span class="gp">In [129]: </span><span class="n">index</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">date_range</span><span class="p">(</span><span class="s2">"10/1/1999"</span><span class="p">,</span> <span class="n">periods</span><span class="o">=</span><span class="mi">1100</span><span class="p">)</span>

<span class="gp">In [130]: </span><span class="n">ts</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">normal</span><span class="p">(</span><span class="mf">0.5</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">1100</span><span class="p">),</span> <span class="n">index</span><span class="p">)</span>

<span class="gp">In [131]: </span><span class="n">ts</span> <span class="o">=</span> <span class="n">ts</span><span class="o">.</span><span class="n">rolling</span><span class="p">(</span><span class="n">window</span><span class="o">=</span><span class="mi">100</span><span class="p">,</span> <span class="n">min_periods</span><span class="o">=</span><span class="mi">100</span><span class="p">)</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span><span class="o">.</span><span class="n">dropna</span><span class="p">()</span>

<span class="gp">In [132]: </span><span class="n">ts</span><span class="o">.</span><span class="n">head</span><span class="p">()</span>
<span class="gh">Out[132]: </span>
<span class="go">2000-01-08    0.779333</span>
<span class="go">2000-01-09    0.778852</span>
<span class="go">2000-01-10    0.786476</span>
<span class="go">2000-01-11    0.782797</span>
<span class="go">2000-01-12    0.798110</span>
<span class="go">Freq: D, dtype: float64</span>

<span class="gp">In [133]: </span><span class="n">ts</span><span class="o">.</span><span class="n">tail</span><span class="p">()</span>
<span class="gh">Out[133]: </span>
<span class="go">2002-09-30    0.660294</span>
<span class="go">2002-10-01    0.631095</span>
<span class="go">2002-10-02    0.673601</span>
<span class="go">2002-10-03    0.709213</span>
<span class="go">2002-10-04    0.719369</span>
<span class="go">Freq: D, dtype: float64</span>

<span class="gp">In [134]: </span><span class="n">transformed</span> <span class="o">=</span> <span class="n">ts</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">year</span><span class="p">)</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="p">(</span><span class="n">x</span> <span class="o">-</span> <span class="n">x</span><span class="o">.</span><span class="n">mean</span><span class="p">())</span> <span class="o">/</span> <span class="n">x</span><span class="o">.</span><span class="n">std</span><span class="p">()</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>
</pre>
</div>
</div>
<p>We would expect the result to now have mean 0 and standard deviation 1 within
each group (up to floating-point error), which we can easily check:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell58"><span></span><span class="go"># Original Data</span>
<span class="gp">In [135]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">ts</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">year</span><span class="p">)</span>

<span class="gp">In [136]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>
<span class="gh">Out[136]: </span>
<span class="go">2000    0.442441</span>
<span class="go">2001    0.526246</span>
<span class="go">2002    0.459365</span>
<span class="go">dtype: float64</span>

<span class="gp">In [137]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">std</span><span class="p">()</span>
<span class="gh">Out[137]: </span>
<span class="go">2000    0.131752</span>
<span class="go">2001    0.210945</span>
<span class="go">2002    0.128753</span>
<span class="go">dtype: float64</span>

<span class="go"># Transformed Data</span>
<span class="gp">In [138]: </span><span class="n">grouped_trans</span> <span class="o">=</span> <span class="n">transformed</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">year</span><span class="p">)</span>

<span class="gp">In [139]: </span><span class="n">grouped_trans</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>
<span class="gh">Out[139]: </span>
<span class="go">2000   -4.870756e-16</span>
<span class="go">2001   -1.545187e-16</span>
<span class="go">2002    4.136282e-16</span>
<span class="go">dtype: float64</span>

<span class="gp">In [140]: </span><span class="n">grouped_trans</span><span class="o">.</span><span class="n">std</span><span class="p">()</span>
<span class="gh">Out[140]: </span>
<span class="go">2000    1.0</span>
<span class="go">2001    1.0</span>
<span class="go">2002    1.0</span>
<span class="go">dtype: float64</span>
</pre>
</div>
</div>
<p>We can also visually compare the original and transformed data sets.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell59"><span></span><span class="gp">In [141]: </span><span class="n">compare</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">({</span><span class="s2">"Original"</span><span class="p">:</span> <span class="n">ts</span><span class="p">,</span> <span class="s2">"Transformed"</span><span class="p">:</span> <span class="n">transformed</span><span class="p">})</span>

<span class="gp">In [142]: </span><span class="n">compare</span><span class="o">.</span><span class="n">plot</span><span class="p">()</span>
<span class="gh">Out[142]: </span><span class="go">&lt;Axes: &gt;</span>
</pre>
</div>
</div>
<img alt="../_images/groupby_transform_plot.png" src="../_images/groupby_transform_plot.png">
<p>Transformation functions that have lower dimension outputs are broadcast to
match the shape of the input array.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell60"><span></span><span class="gp">In [143]: </span><span class="n">ts</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">year</span><span class="p">)</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">max</span><span class="p">()</span> <span class="o">-</span> <span class="n">x</span><span class="o">.</span><span class="n">min</span><span class="p">())</span>
<span class="gh">Out[143]: </span>
<span class="go">2000-01-08    0.623893</span>
<span class="go">2000-01-09    0.623893</span>
<span class="go">2000-01-10    0.623893</span>
<span class="go">2000-01-11    0.623893</span>
<span class="go">2000-01-12    0.623893</span>
<span class="go">                ...   </span>
<span class="go">2002-09-30    0.558275</span>
<span class="go">2002-10-01    0.558275</span>
<span class="go">2002-10-02    0.558275</span>
<span class="go">2002-10-03    0.558275</span>
<span class="go">2002-10-04    0.558275</span>
<span class="go">Freq: D, Length: 1001, dtype: float64</span>
</pre>
</div>
</div>
<p>Another common data transform is to replace missing data with the group mean.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell61"><span></span><span class="gp">In [144]: </span><span class="n">cols</span> <span class="o">=</span> <span class="p">[</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">,</span> <span class="s2">"C"</span><span class="p">]</span>

<span class="gp">In [145]: </span><span class="n">values</span> <span class="o">=</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">1000</span><span class="p">,</span> <span class="mi">3</span><span class="p">)</span>

<span class="gp">In [146]: </span><span class="n">values</span><span class="p">[</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randint</span><span class="p">(</span><span class="mi">0</span><span class="p">,</span> <span class="mi">1000</span><span class="p">,</span> <span class="mi">100</span><span class="p">),</span> <span class="mi">0</span><span class="p">]</span> <span class="o">=</span> <span class="n">np</span><span class="o">.</span><span class="n">nan</span>

<span class="gp">In [147]: </span><span class="n">values</span><span class="p">[</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randint</span><span class="p">(</span><span class="mi">0</span><span class="p">,</span> <span class="mi">1000</span><span class="p">,</span> <span class="mi">50</span><span class="p">),</span> <span class="mi">1</span><span class="p">]</span> <span class="o">=</span> <span class="n">np</span><span class="o">.</span><span class="n">nan</span>

<span class="gp">In [148]: </span><span class="n">values</span><span class="p">[</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randint</span><span class="p">(</span><span class="mi">0</span><span class="p">,</span> <span class="mi">1000</span><span class="p">,</span> <span class="mi">200</span><span class="p">),</span> <span class="mi">2</span><span class="p">]</span> <span class="o">=</span> <span class="n">np</span><span class="o">.</span><span class="n">nan</span>

<span class="gp">In [149]: </span><span class="n">data_df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="n">values</span><span class="p">,</span> <span class="n">columns</span><span class="o">=</span><span class="n">cols</span><span class="p">)</span>

<span class="gp">In [150]: </span><span class="n">data_df</span>
<span class="gh">Out[150]: </span>
<span class="go">            A         B         C</span>
<span class="go">0    1.539708 -1.166480  0.533026</span>
<span class="go">1    1.302092 -0.505754       NaN</span>
<span class="go">2   -0.371983  1.104803 -0.651520</span>
<span class="go">3   -1.309622  1.118697 -1.161657</span>
<span class="go">4   -1.924296  0.396437  0.812436</span>
<span class="go">..        ...       ...       ...</span>
<span class="go">995 -0.093110  0.683847 -0.774753</span>
<span class="go">996 -0.185043  1.438572       NaN</span>
<span class="go">997 -0.394469 -0.642343  0.011374</span>
<span class="go">998 -1.174126  1.857148       NaN</span>
<span class="go">999  0.234564  0.517098  0.393534</span>

<span class="go">[1000 rows x 3 columns]</span>

<span class="gp">In [151]: </span><span class="n">countries</span> <span class="o">=</span> <span class="n">np</span><span class="o">.</span><span class="n">array</span><span class="p">([</span><span class="s2">"US"</span><span class="p">,</span> <span class="s2">"UK"</span><span class="p">,</span> <span class="s2">"GR"</span><span class="p">,</span> <span class="s2">"JP"</span><span class="p">])</span>

<span class="gp">In [152]: </span><span class="n">key</span> <span class="o">=</span> <span class="n">countries</span><span class="p">[</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randint</span><span class="p">(</span><span class="mi">0</span><span class="p">,</span> <span class="mi">4</span><span class="p">,</span> <span class="mi">1000</span><span class="p">)]</span>

<span class="gp">In [153]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">data_df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">key</span><span class="p">)</span>

<span class="go"># Non-NA count in each group</span>
<span class="gp">In [154]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">count</span><span class="p">()</span>
<span class="gh">Out[154]: </span>
<span class="go">      A    B    C</span>
<span class="go">GR  209  217  189</span>
<span class="go">JP  240  255  217</span>
<span class="go">UK  216  231  193</span>
<span class="go">US  239  250  217</span>

<span class="gp">In [155]: </span><span class="n">transformed</span> <span class="o">=</span> <span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">fillna</span><span class="p">(</span><span class="n">x</span><span class="o">.</span><span class="n">mean</span><span class="p">()))</span>
</pre>
</div>
</div>
<p>We can verify that the group means have not changed in the transformed data,
and that the transformed data contains no NAs.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell62"><span></span><span class="gp">In [156]: </span><span class="n">grouped_trans</span> <span class="o">=</span> <span class="n">transformed</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">key</span><span class="p">)</span>

<span class="gp">In [157]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>  <span class="c1"># original group means</span>
<span class="gh">Out[157]: </span>
<span class="go">           A         B         C</span>
<span class="go">GR -0.098371 -0.015420  0.068053</span>
<span class="go">JP  0.069025  0.023100 -0.077324</span>
<span class="go">UK  0.034069 -0.052580 -0.116525</span>
<span class="go">US  0.058664 -0.020399  0.028603</span>

<span class="gp">In [158]: </span><span class="n">grouped_trans</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>  <span class="c1"># transformation did not change group means</span>
<span class="gh">Out[158]: </span>
<span class="go">           A         B         C</span>
<span class="go">GR -0.098371 -0.015420  0.068053</span>
<span class="go">JP  0.069025  0.023100 -0.077324</span>
<span class="go">UK  0.034069 -0.052580 -0.116525</span>
<span class="go">US  0.058664 -0.020399  0.028603</span>

<span class="gp">In [159]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">count</span><span class="p">()</span>  <span class="c1"># original has some missing data points</span>
<span class="gh">Out[159]: </span>
<span class="go">      A    B    C</span>
<span class="go">GR  209  217  189</span>
<span class="go">JP  240  255  217</span>
<span class="go">UK  216  231  193</span>
<span class="go">US  239  250  217</span>

<span class="gp">In [160]: </span><span class="n">grouped_trans</span><span class="o">.</span><span class="n">count</span><span class="p">()</span>  <span class="c1"># counts after transformation</span>
<span class="gh">Out[160]: </span>
<span class="go">      A    B    C</span>
<span class="go">GR  228  228  228</span>
<span class="go">JP  267  267  267</span>
<span class="go">UK  247  247  247</span>
<span class="go">US  258  258  258</span>

<span class="gp">In [161]: </span><span class="n">grouped_trans</span><span class="o">.</span><span class="n">size</span><span class="p">()</span>  <span class="c1"># Verify non-NA count equals group size</span>
<span class="gh">Out[161]: </span>
<span class="go">GR    228</span>
<span class="go">JP    267</span>
<span class="go">UK    247</span>
<span class="go">US    258</span>
<span class="go">dtype: int64</span>
</pre>
</div>
</div>
<p id="groupby-efficient-transforms">As mentioned in the note above, each of the examples in this section can be computed
more efficiently using built-in methods. In the code below, the inefficient way
using a UDF is commented out and the faster alternative appears below.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell63"><span></span><span class="go"># result = ts.groupby(lambda x: x.year).transform(</span>
<span class="go">#     lambda x: (x - x.mean()) / x.std()</span>
<span class="go"># )</span>
<span class="gp">In [162]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">ts</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">year</span><span class="p">)</span>

<span class="gp">In [163]: </span><span class="n">result</span> <span class="o">=</span> <span class="p">(</span><span class="n">ts</span> <span class="o">-</span> <span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="s2">"mean"</span><span class="p">))</span> <span class="o">/</span> <span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="s2">"std"</span><span class="p">)</span>

<span class="go"># result = ts.groupby(lambda x: x.year).transform(lambda x: x.max() - x.min())</span>
<span class="gp">In [164]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">ts</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">year</span><span class="p">)</span>

<span class="gp">In [165]: </span><span class="n">result</span> <span class="o">=</span> <span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="s2">"max"</span><span class="p">)</span> <span class="o">-</span> <span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="s2">"min"</span><span class="p">)</span>

<span class="go"># grouped = data_df.groupby(key)</span>
<span class="go"># result = grouped.transform(lambda x: x.fillna(x.mean()))</span>
<span class="gp">In [166]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">data_df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">key</span><span class="p">)</span>

<span class="gp">In [167]: </span><span class="n">result</span> <span class="o">=</span> <span class="n">data_df</span><span class="o">.</span><span class="n">fillna</span><span class="p">(</span><span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="s2">"mean"</span><span class="p">))</span>
</pre>
</div>
</div>
</section>
<section id="window-and-resample-operations">
<span id="groupby-transform-window-resample"></span><h3>Window and resample operations<a class="headerlink" href="#window-and-resample-operations" title="Link to this heading">#</a></h3>
<p>It is possible to use <code class="docutils literal notranslate"><span class="pre">resample()</span></code>, <code class="docutils literal notranslate"><span class="pre">expanding()</span></code> and
<code class="docutils literal notranslate"><span class="pre">rolling()</span></code> as methods on groupbys.</p>
<p>The example below will apply the <code class="docutils literal notranslate"><span class="pre">rolling()</span></code> method on the samples of
the column B, based on the groups of column A.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell64"><span></span><span class="gp">In [168]: </span><span class="n">df_re</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">({</span><span class="s2">"A"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">]</span> <span class="o">*</span> <span class="mi">10</span> <span class="o">+</span> <span class="p">[</span><span class="mi">5</span><span class="p">]</span> <span class="o">*</span> <span class="mi">10</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">arange</span><span class="p">(</span><span class="mi">20</span><span class="p">)})</span>

<span class="gp">In [169]: </span><span class="n">df_re</span>
<span class="gh">Out[169]: </span>
<span class="go">    A   B</span>
<span class="go">0   1   0</span>
<span class="go">1   1   1</span>
<span class="go">2   1   2</span>
<span class="go">3   1   3</span>
<span class="go">4   1   4</span>
<span class="go">.. ..  ..</span>
<span class="go">15  5  15</span>
<span class="go">16  5  16</span>
<span class="go">17  5  17</span>
<span class="go">18  5  18</span>
<span class="go">19  5  19</span>

<span class="go">[20 rows x 2 columns]</span>

<span class="gp">In [170]: </span><span class="n">df_re</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">rolling</span><span class="p">(</span><span class="mi">4</span><span class="p">)</span><span class="o">.</span><span class="n">B</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>
<span class="gh">Out[170]: </span>
<span class="go">A    </span>
<span class="go">1  0      NaN</span>
<span class="go">   1      NaN</span>
<span class="go">   2      NaN</span>
<span class="go">   3      1.5</span>
<span class="go">   4      2.5</span>
<span class="go">         ... </span>
<span class="go">5  15    13.5</span>
<span class="go">   16    14.5</span>
<span class="go">   17    15.5</span>
<span class="go">   18    16.5</span>
<span class="go">   19    17.5</span>
<span class="go">Name: B, Length: 20, dtype: float64</span>
</pre>
</div>
</div>
<p>The <code class="docutils literal notranslate"><span class="pre">expanding()</span></code> method will accumulate a given operation
(<code class="docutils literal notranslate"><span class="pre">sum()</span></code> in the example) for all the members of each particular
group.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell65"><span></span><span class="gp">In [171]: </span><span class="n">df_re</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">expanding</span><span class="p">()</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[171]: </span>
<span class="go">          B</span>
<span class="go">A          </span>
<span class="go">1 0     0.0</span>
<span class="go">  1     1.0</span>
<span class="go">  2     3.0</span>
<span class="go">  3     6.0</span>
<span class="go">  4    10.0</span>
<span class="go">...     ...</span>
<span class="go">5 15   75.0</span>
<span class="go">  16   91.0</span>
<span class="go">  17  108.0</span>
<span class="go">  18  126.0</span>
<span class="go">  19  145.0</span>

<span class="go">[20 rows x 1 columns]</span>
</pre>
</div>
</div>
<p>Suppose you want to use the <code class="docutils literal notranslate"><span class="pre">resample()</span></code> method to get a daily
frequency in each group of your dataframe, and wish to complete the
missing values with the <code class="docutils literal notranslate"><span class="pre">ffill()</span></code> method.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell66"><span></span><span class="gp">In [172]: </span><span class="n">df_re</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="p">{</span>
<span class="gp">   .....: </span>        <span class="s2">"date"</span><span class="p">:</span> <span class="n">pd</span><span class="o">.</span><span class="n">date_range</span><span class="p">(</span><span class="n">start</span><span class="o">=</span><span class="s2">"2016-01-01"</span><span class="p">,</span> <span class="n">periods</span><span class="o">=</span><span class="mi">4</span><span class="p">,</span> <span class="n">freq</span><span class="o">=</span><span class="s2">"W"</span><span class="p">),</span>
<span class="gp">   .....: </span>        <span class="s2">"group"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span>
<span class="gp">   .....: </span>        <span class="s2">"val"</span><span class="p">:</span> <span class="p">[</span><span class="mi">5</span><span class="p">,</span> <span class="mi">6</span><span class="p">,</span> <span class="mi">7</span><span class="p">,</span> <span class="mi">8</span><span class="p">],</span>
<span class="gp">   .....: </span>    <span class="p">}</span>
<span class="gp">   .....: </span><span class="p">)</span><span class="o">.</span><span class="n">set_index</span><span class="p">(</span><span class="s2">"date"</span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [173]: </span><span class="n">df_re</span>
<span class="gh">Out[173]: </span>
<span class="go">            group  val</span>
<span class="go">date                  </span>
<span class="go">2016-01-03      1    5</span>
<span class="go">2016-01-10      1    6</span>
<span class="go">2016-01-17      2    7</span>
<span class="go">2016-01-24      2    8</span>

<span class="gp">In [174]: </span><span class="n">df_re</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"group"</span><span class="p">)</span><span class="o">.</span><span class="n">resample</span><span class="p">(</span><span class="s2">"1D"</span><span class="p">,</span> <span class="n">include_groups</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span><span class="o">.</span><span class="n">ffill</span><span class="p">()</span>
<span class="gh">Out[174]: </span>
<span class="go">                  val</span>
<span class="go">group date           </span>
<span class="go">1     2016-01-03    5</span>
<span class="go">      2016-01-04    5</span>
<span class="go">      2016-01-05    5</span>
<span class="go">      2016-01-06    5</span>
<span class="go">      2016-01-07    5</span>
<span class="go">...               ...</span>
<span class="go">2     2016-01-20    7</span>
<span class="go">      2016-01-21    7</span>
<span class="go">      2016-01-22    7</span>
<span class="go">      2016-01-23    7</span>
<span class="go">      2016-01-24    8</span>

<span class="go">[16 rows x 1 columns]</span>
</pre>
</div>
</div>
</section>
</section>
<section id="filtration">
<span id="groupby-filter"></span><h2>Filtration<a class="headerlink" href="#filtration" title="Link to this heading">#</a></h2>
<p>A filtration is a GroupBy operation that subsets the original grouping object. It
may either filter out entire groups, part of groups, or both. Filtrations return
a filtered version of the calling object, including the grouping columns when provided.
In the following example, <code class="docutils literal notranslate"><span class="pre">class</span></code> is included in the result.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell67"><span></span><span class="gp">In [175]: </span><span class="n">speeds</span>
<span class="gh">Out[175]: </span>
<span class="go">          class           order  max_speed</span>
<span class="go">falcon     bird   Falconiformes      389.0</span>
<span class="go">parrot     bird  Psittaciformes       24.0</span>
<span class="go">lion     mammal       Carnivora       80.2</span>
<span class="go">monkey   mammal        Primates        NaN</span>
<span class="go">leopard  mammal       Carnivora       58.0</span>

<span class="gp">In [176]: </span><span class="n">speeds</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"class"</span><span class="p">)</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[176]: </span>
<span class="go">         class           order  max_speed</span>
<span class="go">parrot    bird  Psittaciformes       24.0</span>
<span class="go">monkey  mammal        Primates        NaN</span>
</pre>
</div>
</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Unlike aggregations, filtrations do not add the group keys to the index of the
result. Because of this, passing <code class="docutils literal notranslate"><span class="pre">as_index=False</span></code> or <code class="docutils literal notranslate"><span class="pre">sort=True</span></code> will not
affect these methods.</p>
</div>
<p>Filtrations will respect subsetting the columns of the GroupBy object.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell68"><span></span><span class="gp">In [177]: </span><span class="n">speeds</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"class"</span><span class="p">)[[</span><span class="s2">"order"</span><span class="p">,</span> <span class="s2">"max_speed"</span><span class="p">]]</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[177]: </span>
<span class="go">                 order  max_speed</span>
<span class="go">parrot  Psittaciformes       24.0</span>
<span class="go">monkey        Primates        NaN</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell68">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<section id="built-in-filtrations">
<h3>Built-in filtrations<a class="headerlink" href="#built-in-filtrations" title="Link to this heading">#</a></h3>
<p>The following methods on GroupBy act as filtrations. All these methods have an
efficient, GroupBy-specific, implementation.</p>
<table class="table">
<colgroup>
<col style="width: 20.0%">
<col style="width: 80.0%">
</colgroup>
<thead>
<tr class="row-odd"><th class="head"><p>Method</p></th>
<th class="head"><p>Description</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.head.html#pandas.core.groupby.DataFrameGroupBy.head" title="pandas.core.groupby.DataFrameGroupBy.head"><code><span class="pre">head()</span></code></a></p></td>
<td><p>Select the top row(s) of each group</p></td>
</tr>
<tr class="row-odd"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.nth.html#pandas.core.groupby.DataFrameGroupBy.nth" title="pandas.core.groupby.DataFrameGroupBy.nth"><code><span class="pre">nth()</span></code></a></p></td>
<td><p>Select the nth row(s) of each group</p></td>
</tr>
<tr class="row-even"><td><p><a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.tail.html#pandas.core.groupby.DataFrameGroupBy.tail" title="pandas.core.groupby.DataFrameGroupBy.tail"><code><span class="pre">tail()</span></code></a></p></td>
<td><p>Select the bottom row(s) of each group</p></td>
</tr>
</tbody>
</table>
<p>Users can also use transformations along with Boolean indexing to construct complex
filtrations within groups. For example, suppose we are given groups of products and
their volumes, and we wish to subset the data to only the largest products capturing no
more than 90% of the total volume within each group.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell69"><span></span><span class="gp">In [178]: </span><span class="n">product_volumes</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="p">{</span>
<span class="gp">   .....: </span>        <span class="s2">"group"</span><span class="p">:</span> <span class="nb">list</span><span class="p">(</span><span class="s2">"xxxxyyy"</span><span class="p">),</span>
<span class="gp">   .....: </span>        <span class="s2">"product"</span><span class="p">:</span> <span class="nb">list</span><span class="p">(</span><span class="s2">"abcdefg"</span><span class="p">),</span>
<span class="gp">   .....: </span>        <span class="s2">"volume"</span><span class="p">:</span> <span class="p">[</span><span class="mi">10</span><span class="p">,</span> <span class="mi">30</span><span class="p">,</span> <span class="mi">20</span><span class="p">,</span> <span class="mi">15</span><span class="p">,</span> <span class="mi">40</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="mi">20</span><span class="p">],</span>
<span class="gp">   .....: </span>    <span class="p">}</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [179]: </span><span class="n">product_volumes</span>
<span class="gh">Out[179]: </span>
<span class="go">  group product  volume</span>
<span class="go">0     x       a      10</span>
<span class="go">1     x       b      30</span>
<span class="go">2     x       c      20</span>
<span class="go">3     x       d      15</span>
<span class="go">4     y       e      40</span>
<span class="go">5     y       f      10</span>
<span class="go">6     y       g      20</span>

<span class="go"># Sort by volume to select the largest products first</span>
<span class="gp">In [180]: </span><span class="n">product_volumes</span> <span class="o">=</span> <span class="n">product_volumes</span><span class="o">.</span><span class="n">sort_values</span><span class="p">(</span><span class="s2">"volume"</span><span class="p">,</span> <span class="n">ascending</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>

<span class="gp">In [181]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">product_volumes</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"group"</span><span class="p">)[</span><span class="s2">"volume"</span><span class="p">]</span>

<span class="gp">In [182]: </span><span class="n">cumpct</span> <span class="o">=</span> <span class="n">grouped</span><span class="o">.</span><span class="n">cumsum</span><span class="p">()</span> <span class="o">/</span> <span class="n">grouped</span><span class="o">.</span><span class="n">transform</span><span class="p">(</span><span class="s2">"sum"</span><span class="p">)</span>

<span class="gp">In [183]: </span><span class="n">cumpct</span>
<span class="gh">Out[183]: </span>
<span class="go">4    0.571429</span>
<span class="go">1    0.400000</span>
<span class="go">2    0.666667</span>
<span class="go">6    0.857143</span>
<span class="go">3    0.866667</span>
<span class="go">0    1.000000</span>
<span class="go">5    1.000000</span>
<span class="go">Name: volume, dtype: float64</span>

<span class="gp">In [184]: </span><span class="n">significant_products</span> <span class="o">=</span> <span class="n">product_volumes</span><span class="p">[</span><span class="n">cumpct</span> <span class="o">&lt;=</span> <span class="mf">0.9</span><span class="p">]</span>

<span class="gp">In [185]: </span><span class="n">significant_products</span><span class="o">.</span><span class="n">sort_values</span><span class="p">([</span><span class="s2">"group"</span><span class="p">,</span> <span class="s2">"product"</span><span class="p">])</span>
<span class="gh">Out[185]: </span>
<span class="go">  group product  volume</span>
<span class="go">1     x       b      30</span>
<span class="go">2     x       c      20</span>
<span class="go">3     x       d      15</span>
<span class="go">4     y       e      40</span>
<span class="go">6     y       g      20</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell69">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="the-filter-method">
<h3>The <code><span class="pre">filter</span></code> method<a class="headerlink" href="#the-filter-method" title="Link to this heading">#</a></h3>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Filtering by supplying <code class="docutils literal notranslate"><span class="pre">filter</span></code> with a User-Defined Function (UDF) is
often less performant than using the built-in methods on GroupBy.
Consider breaking up a complex operation into a chain of operations that utilize
the built-in methods.</p>
</div>
<p>The <code class="docutils literal notranslate"><span class="pre">filter</span></code> method takes a User-Defined Function (UDF) that, when applied to
an entire group, returns either <code class="docutils literal notranslate"><span class="pre">True</span></code> or <code class="docutils literal notranslate"><span class="pre">False</span></code>. The result of the <code class="docutils literal notranslate"><span class="pre">filter</span></code>
method is then the subset of groups for which the UDF returned <code class="docutils literal notranslate"><span class="pre">True</span></code>.</p>
<p>Suppose we want to take only elements that belong to groups with a group sum greater
than 2.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell70"><span></span><span class="gp">In [186]: </span><span class="n">sf</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="mi">3</span><span class="p">])</span>

<span class="gp">In [187]: </span><span class="n">sf</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">sf</span><span class="p">)</span><span class="o">.</span><span class="n">filter</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span> <span class="o">&gt;</span> <span class="mi">2</span><span class="p">)</span>
<span class="gh">Out[187]: </span>
<span class="go">3    3</span>
<span class="go">4    3</span>
<span class="go">5    3</span>
<span class="go">dtype: int64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell70">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>Another useful operation is filtering out elements that belong to groups
with only a couple members.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell71"><span></span><span class="gp">In [188]: </span><span class="n">dff</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">({</span><span class="s2">"A"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">arange</span><span class="p">(</span><span class="mi">8</span><span class="p">),</span> <span class="s2">"B"</span><span class="p">:</span> <span class="nb">list</span><span class="p">(</span><span class="s2">"aabbbbcc"</span><span class="p">)})</span>

<span class="gp">In [189]: </span><span class="n">dff</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"B"</span><span class="p">)</span><span class="o">.</span><span class="n">filter</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="nb">len</span><span class="p">(</span><span class="n">x</span><span class="p">)</span> <span class="o">&gt;</span> <span class="mi">2</span><span class="p">)</span>
<span class="gh">Out[189]: </span>
<span class="go">   A  B</span>
<span class="go">2  2  b</span>
<span class="go">3  3  b</span>
<span class="go">4  4  b</span>
<span class="go">5  5  b</span>
</pre>
</div>
</div>
<p>Alternatively, instead of dropping the offending groups, we can return a
like-indexed objects where the groups that do not pass the filter are filled
with NaNs.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell72"><span></span><span class="gp">In [190]: </span><span class="n">dff</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"B"</span><span class="p">)</span><span class="o">.</span><span class="n">filter</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="nb">len</span><span class="p">(</span><span class="n">x</span><span class="p">)</span> <span class="o">&gt;</span> <span class="mi">2</span><span class="p">,</span> <span class="n">dropna</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="gh">Out[190]: </span>
<span class="go">     A    B</span>
<span class="go">0  NaN  NaN</span>
<span class="go">1  NaN  NaN</span>
<span class="go">2  2.0    b</span>
<span class="go">3  3.0    b</span>
<span class="go">4  4.0    b</span>
<span class="go">5  5.0    b</span>
<span class="go">6  NaN  NaN</span>
<span class="go">7  NaN  NaN</span>
</pre>
</div>
</div>
<p>For DataFrames with multiple columns, filters should explicitly specify a column as the filter criterion.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell73"><span></span><span class="gp">In [191]: </span><span class="n">dff</span><span class="p">[</span><span class="s2">"C"</span><span class="p">]</span> <span class="o">=</span> <span class="n">np</span><span class="o">.</span><span class="n">arange</span><span class="p">(</span><span class="mi">8</span><span class="p">)</span>

<span class="gp">In [192]: </span><span class="n">dff</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"B"</span><span class="p">)</span><span class="o">.</span><span class="n">filter</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="nb">len</span><span class="p">(</span><span class="n">x</span><span class="p">[</span><span class="s2">"C"</span><span class="p">])</span> <span class="o">&gt;</span> <span class="mi">2</span><span class="p">)</span>
<span class="gh">Out[192]: </span>
<span class="go">   A  B  C</span>
<span class="go">2  2  b  2</span>
<span class="go">3  3  b  3</span>
<span class="go">4  4  b  4</span>
<span class="go">5  5  b  5</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell73">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
</section>
<section id="flexible-apply">
<span id="groupby-apply"></span><h2>Flexible <code class="docutils literal notranslate"><span class="pre">apply</span></code><a class="headerlink" href="#flexible-apply" title="Link to this heading">#</a></h2>
<p>Some operations on the grouped data might not fit into the aggregation,
transformation, or filtration categories. For these, you can use the <code class="docutils literal notranslate"><span class="pre">apply</span></code>
function.</p>
<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p><code class="docutils literal notranslate"><span class="pre">apply</span></code> has to try to infer from the result whether it should act as a reducer,
transformer, <em>or</em> filter, depending on exactly what is passed to it. Thus the
grouped column(s) may be included in the output or not. While
it tries to intelligently guess how to behave, it can sometimes guess wrong.</p>
</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>All of the examples in this section can be more reliably, and more efficiently,
computed using other pandas functionality.</p>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell74"><span></span><span class="gp">In [193]: </span><span class="n">df</span>
<span class="gh">Out[193]: </span>
<span class="go">     A      B         C         D</span>
<span class="go">0  foo    one -0.575247  1.346061</span>
<span class="go">1  bar    one  0.254161  1.511763</span>
<span class="go">2  foo    two -1.143704  1.627081</span>
<span class="go">3  bar  three  0.215897 -0.990582</span>
<span class="go">4  foo    two  1.193555 -0.441652</span>
<span class="go">5  bar    two -0.077118  1.211526</span>
<span class="go">6  foo    one -0.408530  0.268520</span>
<span class="go">7  foo  three -0.862495  0.024580</span>

<span class="gp">In [194]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span>

<span class="go"># could also just call .describe()</span>
<span class="gp">In [195]: </span><span class="n">grouped</span><span class="p">[</span><span class="s2">"C"</span><span class="p">]</span><span class="o">.</span><span class="n">apply</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="o">.</span><span class="n">describe</span><span class="p">())</span>
<span class="gh">Out[195]: </span>
<span class="go">A         </span>
<span class="go">bar  count    3.000000</span>
<span class="go">     mean     0.130980</span>
<span class="go">     std      0.181231</span>
<span class="go">     min     -0.077118</span>
<span class="go">     25%      0.069390</span>
<span class="go">                ...   </span>
<span class="go">foo  min     -1.143704</span>
<span class="go">     25%     -0.862495</span>
<span class="go">     50%     -0.575247</span>
<span class="go">     75%     -0.408530</span>
<span class="go">     max      1.193555</span>
<span class="go">Name: C, Length: 16, dtype: float64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell74">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>The dimension of the returned result can also change:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell75"><span></span><span class="gp">In [196]: </span><span class="n">grouped</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s1">'A'</span><span class="p">)[</span><span class="s1">'C'</span><span class="p">]</span>

<span class="gp">In [197]: </span><span class="k">def</span><span class="w"> </span><span class="nf">f</span><span class="p">(</span><span class="n">group</span><span class="p">):</span>
<span class="gp">   .....: </span>    <span class="k">return</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">({</span><span class="s1">'original'</span><span class="p">:</span> <span class="n">group</span><span class="p">,</span>
<span class="gp">   .....: </span>                         <span class="s1">'demeaned'</span><span class="p">:</span> <span class="n">group</span> <span class="o">-</span> <span class="n">group</span><span class="o">.</span><span class="n">mean</span><span class="p">()})</span>
<span class="gp">   .....: </span>

<span class="gp">In [198]: </span><span class="n">grouped</span><span class="o">.</span><span class="n">apply</span><span class="p">(</span><span class="n">f</span><span class="p">)</span>
<span class="gh">Out[198]: </span>
<span class="go">       original  demeaned</span>
<span class="go">A                        </span>
<span class="go">bar 1  0.254161  0.123181</span>
<span class="go">    3  0.215897  0.084917</span>
<span class="go">    5 -0.077118 -0.208098</span>
<span class="go">foo 0 -0.575247 -0.215962</span>
<span class="go">    2 -1.143704 -0.784420</span>
<span class="go">    4  1.193555  1.552839</span>
<span class="go">    6 -0.408530 -0.049245</span>
<span class="go">    7 -0.862495 -0.503211</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell75">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p><code class="docutils literal notranslate"><span class="pre">apply</span></code> on a Series can operate on a returned value from the applied function
that is itself a series, and possibly upcast the result to a DataFrame:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell76"><span></span><span class="gp">In [199]: </span><span class="k">def</span><span class="w"> </span><span class="nf">f</span><span class="p">(</span><span class="n">x</span><span class="p">):</span>
<span class="gp">   .....: </span>    <span class="k">return</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="n">x</span><span class="p">,</span> <span class="n">x</span> <span class="o">**</span> <span class="mi">2</span><span class="p">],</span> <span class="n">index</span><span class="o">=</span><span class="p">[</span><span class="s2">"x"</span><span class="p">,</span> <span class="s2">"x^2"</span><span class="p">])</span>
<span class="gp">   .....: </span>

<span class="gp">In [200]: </span><span class="n">s</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">rand</span><span class="p">(</span><span class="mi">5</span><span class="p">))</span>

<span class="gp">In [201]: </span><span class="n">s</span>
<span class="gh">Out[201]: </span>
<span class="go">0    0.582898</span>
<span class="go">1    0.098352</span>
<span class="go">2    0.001438</span>
<span class="go">3    0.009420</span>
<span class="go">4    0.815826</span>
<span class="go">dtype: float64</span>

<span class="gp">In [202]: </span><span class="n">s</span><span class="o">.</span><span class="n">apply</span><span class="p">(</span><span class="n">f</span><span class="p">)</span>
<span class="gh">Out[202]: </span>
<span class="go">          x       x^2</span>
<span class="go">0  0.582898  0.339770</span>
<span class="go">1  0.098352  0.009673</span>
<span class="go">2  0.001438  0.000002</span>
<span class="go">3  0.009420  0.000089</span>
<span class="go">4  0.815826  0.665572</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell76">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>Similar to <a href="#groupby-aggregate-agg"><span class="std std-ref">The aggregate() method</span></a>, the resulting dtype will reflect that of the
apply function. If the results from different groups have different dtypes, then
a common dtype will be determined in the same way as <code class="docutils literal notranslate"><span class="pre">DataFrame</span></code> construction.</p>
<section id="control-grouped-column-s-placement-with-group-keys">
<h3>Control grouped column(s) placement with <code class="docutils literal notranslate"><span class="pre">group_keys</span></code><a class="headerlink" href="#control-grouped-column-s-placement-with-group-keys" title="Link to this heading">#</a></h3>
<p>To control whether the grouped column(s) are included in the indices, you can use
the argument <code class="docutils literal notranslate"><span class="pre">group_keys</span></code> which defaults to <code class="docutils literal notranslate"><span class="pre">True</span></code>. Compare</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell77"><span></span><span class="gp">In [203]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">,</span> <span class="n">group_keys</span><span class="o">=</span><span class="kc">True</span><span class="p">)</span><span class="o">.</span><span class="n">apply</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="p">,</span> <span class="n">include_groups</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="gh">Out[203]: </span>
<span class="go">           B         C         D</span>
<span class="go">A                               </span>
<span class="go">bar 1    one  0.254161  1.511763</span>
<span class="go">    3  three  0.215897 -0.990582</span>
<span class="go">    5    two -0.077118  1.211526</span>
<span class="go">foo 0    one -0.575247  1.346061</span>
<span class="go">    2    two -1.143704  1.627081</span>
<span class="go">    4    two  1.193555 -0.441652</span>
<span class="go">    6    one -0.408530  0.268520</span>
<span class="go">    7  three -0.862495  0.024580</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell77">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>with</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell78"><span></span><span class="gp">In [204]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">,</span> <span class="n">group_keys</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span><span class="o">.</span><span class="n">apply</span><span class="p">(</span><span class="k">lambda</span> <span class="n">x</span><span class="p">:</span> <span class="n">x</span><span class="p">,</span> <span class="n">include_groups</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="gh">Out[204]: </span>
<span class="go">       B         C         D</span>
<span class="go">0    one -0.575247  1.346061</span>
<span class="go">1    one  0.254161  1.511763</span>
<span class="go">2    two -1.143704  1.627081</span>
<span class="go">3  three  0.215897 -0.990582</span>
<span class="go">4    two  1.193555 -0.441652</span>
<span class="go">5    two -0.077118  1.211526</span>
<span class="go">6    one -0.408530  0.268520</span>
<span class="go">7  three -0.862495  0.024580</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell78">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
</section>
<section id="numba-accelerated-routines">
<h2>Numba Accelerated Routines<a class="headerlink" href="#numba-accelerated-routines" title="Link to this heading">#</a></h2>
<div class="versionadded">
<p><span class="versionmodified added">Added in version 1.1.</span></p>
</div>
<p>If <a class="reference external" href="https://numba.pydata.org/">Numba</a> is installed as an optional dependency, the <code class="docutils literal notranslate"><span class="pre">transform</span></code> and
<code class="docutils literal notranslate"><span class="pre">aggregate</span></code> methods support <code class="docutils literal notranslate"><span class="pre">engine='numba'</span></code> and <code class="docutils literal notranslate"><span class="pre">engine_kwargs</span></code> arguments.
See <a href="enhancingperf.html#enhancingperf-numba"><span class="std std-ref">enhancing performance with Numba</span></a> for general usage of the arguments
and performance considerations.</p>
<p>The function signature must start with <code class="docutils literal notranslate"><span class="pre">values,</span> <span class="pre">index</span></code> <strong>exactly</strong> as the data belonging to each group
will be passed into <code class="docutils literal notranslate"><span class="pre">values</span></code>, and the group index will be passed into <code class="docutils literal notranslate"><span class="pre">index</span></code>.</p>
<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p>When using <code class="docutils literal notranslate"><span class="pre">engine='numba'</span></code>, there will be no “fall back” behavior internally. The group
data and group index will be passed as NumPy arrays to the JITed user defined function, and no
alternative execution attempts will be tried.</p>
</div>
</section>
<section id="other-useful-features">
<h2>Other useful features<a class="headerlink" href="#other-useful-features" title="Link to this heading">#</a></h2>
<section id="exclusion-of-non-numeric-columns">
<h3>Exclusion of non-numeric columns<a class="headerlink" href="#exclusion-of-non-numeric-columns" title="Link to this heading">#</a></h3>
<p>Again consider the example DataFrame we’ve been looking at:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell79"><span></span><span class="gp">In [205]: </span><span class="n">df</span>
<span class="gh">Out[205]: </span>
<span class="go">     A      B         C         D</span>
<span class="go">0  foo    one -0.575247  1.346061</span>
<span class="go">1  bar    one  0.254161  1.511763</span>
<span class="go">2  foo    two -1.143704  1.627081</span>
<span class="go">3  bar  three  0.215897 -0.990582</span>
<span class="go">4  foo    two  1.193555 -0.441652</span>
<span class="go">5  bar    two -0.077118  1.211526</span>
<span class="go">6  foo    one -0.408530  0.268520</span>
<span class="go">7  foo  three -0.862495  0.024580</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell79">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>Suppose we wish to compute the standard deviation grouped by the <code class="docutils literal notranslate"><span class="pre">A</span></code>
column. There is a slight problem, namely that we don’t care about the data in
column <code class="docutils literal notranslate"><span class="pre">B</span></code> because it is not numeric. You can avoid non-numeric columns by
specifying <code class="docutils literal notranslate"><span class="pre">numeric_only=True</span></code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell80"><span></span><span class="gp">In [206]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">std</span><span class="p">(</span><span class="n">numeric_only</span><span class="o">=</span><span class="kc">True</span><span class="p">)</span>
<span class="gh">Out[206]: </span>
<span class="go">            C         D</span>
<span class="go">A                      </span>
<span class="go">bar  0.181231  1.366330</span>
<span class="go">foo  0.912265  0.884785</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell80">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>Note that <code class="docutils literal notranslate"><span class="pre">df.groupby('A').colname.std().</span></code> is more efficient than
<code class="docutils literal notranslate"><span class="pre">df.groupby('A').std().colname</span></code>. So if the result of an aggregation function
is only needed over one column (here <code class="docutils literal notranslate"><span class="pre">colname</span></code>), it may be filtered
<em>before</em> applying the aggregation function.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell81"><span></span><span class="gp">In [207]: </span><span class="kn">from</span><span class="w"> </span><span class="nn">decimal</span><span class="w"> </span><span class="kn">import</span> <span class="n">Decimal</span>

<span class="gp">In [208]: </span><span class="n">df_dec</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="p">{</span>
<span class="gp">   .....: </span>        <span class="s2">"id"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span>
<span class="gp">   .....: </span>        <span class="s2">"int_column"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="mi">4</span><span class="p">],</span>
<span class="gp">   .....: </span>        <span class="s2">"dec_column"</span><span class="p">:</span> <span class="p">[</span>
<span class="gp">   .....: </span>            <span class="n">Decimal</span><span class="p">(</span><span class="s2">"0.50"</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">Decimal</span><span class="p">(</span><span class="s2">"0.15"</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">Decimal</span><span class="p">(</span><span class="s2">"0.25"</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">Decimal</span><span class="p">(</span><span class="s2">"0.40"</span><span class="p">),</span>
<span class="gp">   .....: </span>        <span class="p">],</span>
<span class="gp">   .....: </span>    <span class="p">}</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [209]: </span><span class="n">df_dec</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"id"</span><span class="p">])[[</span><span class="s2">"dec_column"</span><span class="p">]]</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[209]: </span>
<span class="go">   dec_column</span>
<span class="go">id           </span>
<span class="go">1        0.75</span>
<span class="go">2        0.55</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell81">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="handling-of-un-observed-categorical-values">
<span id="groupby-observed"></span><h3>Handling of (un)observed Categorical values<a class="headerlink" href="#handling-of-un-observed-categorical-values" title="Link to this heading">#</a></h3>
<p>When using a <code class="docutils literal notranslate"><span class="pre">Categorical</span></code> grouper (as a single grouper, or as part of multiple groupers), the <code class="docutils literal notranslate"><span class="pre">observed</span></code> keyword
controls whether to return a cartesian product of all possible groupers values (<code class="docutils literal notranslate"><span class="pre">observed=False</span></code>) or only those
that are observed groupers (<code class="docutils literal notranslate"><span class="pre">observed=True</span></code>).</p>
<p>Show all values:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell82"><span></span><span class="gp">In [210]: </span><span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">])</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">pd</span><span class="o">.</span><span class="n">Categorical</span><span class="p">([</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"a"</span><span class="p">,</span> <span class="s2">"a"</span><span class="p">],</span> <span class="n">categories</span><span class="o">=</span><span class="p">[</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"b"</span><span class="p">]),</span> <span class="n">observed</span><span class="o">=</span><span class="kc">False</span>
<span class="gp">   .....: </span><span class="p">)</span><span class="o">.</span><span class="n">count</span><span class="p">()</span>
<span class="gp">   .....: </span>
<span class="gh">Out[210]: </span>
<span class="go">a    3</span>
<span class="go">b    0</span>
<span class="go">dtype: int64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell82">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>Show only the observed values:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell83"><span></span><span class="gp">In [211]: </span><span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">])</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">pd</span><span class="o">.</span><span class="n">Categorical</span><span class="p">([</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"a"</span><span class="p">,</span> <span class="s2">"a"</span><span class="p">],</span> <span class="n">categories</span><span class="o">=</span><span class="p">[</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"b"</span><span class="p">]),</span> <span class="n">observed</span><span class="o">=</span><span class="kc">True</span>
<span class="gp">   .....: </span><span class="p">)</span><span class="o">.</span><span class="n">count</span><span class="p">()</span>
<span class="gp">   .....: </span>
<span class="gh">Out[211]: </span>
<span class="go">a    3</span>
<span class="go">dtype: int64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell83">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>The returned dtype of the grouped will <em>always</em> include <em>all</em> of the categories that were grouped.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell84"><span></span><span class="gp">In [212]: </span><span class="n">s</span> <span class="o">=</span> <span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">])</span>
<span class="gp">   .....: </span>    <span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">pd</span><span class="o">.</span><span class="n">Categorical</span><span class="p">([</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"a"</span><span class="p">,</span> <span class="s2">"a"</span><span class="p">],</span> <span class="n">categories</span><span class="o">=</span><span class="p">[</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"b"</span><span class="p">]),</span> <span class="n">observed</span><span class="o">=</span><span class="kc">True</span><span class="p">)</span>
<span class="gp">   .....: </span>    <span class="o">.</span><span class="n">count</span><span class="p">()</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [213]: </span><span class="n">s</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">dtype</span>
<span class="gh">Out[213]: </span><span class="go">CategoricalDtype(categories=['a', 'b'], ordered=False, categories_dtype=object)</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell84">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="na-group-handling">
<span id="groupby-missing"></span><h3>NA group handling<a class="headerlink" href="#na-group-handling" title="Link to this heading">#</a></h3>
<p>By <code class="docutils literal notranslate"><span class="pre">NA</span></code>, we are referring to any <code class="docutils literal notranslate"><span class="pre">NA</span></code> values, including
<a href="../reference/api/pandas.NA.html#pandas.NA" title="pandas.NA"><code><span class="pre">NA</span></code></a>, <code class="docutils literal notranslate"><span class="pre">NaN</span></code>, <code class="docutils literal notranslate"><span class="pre">NaT</span></code>, and <code class="docutils literal notranslate"><span class="pre">None</span></code>. If there are any <code class="docutils literal notranslate"><span class="pre">NA</span></code> values in the
grouping key, by default these will be excluded. In other words, any
“<code class="docutils literal notranslate"><span class="pre">NA</span></code> group” will be dropped. You can include NA groups by specifying <code class="docutils literal notranslate"><span class="pre">dropna=False</span></code>.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell85"><span></span><span class="gp">In [214]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">({</span><span class="s2">"key"</span><span class="p">:</span> <span class="p">[</span><span class="mf">1.0</span><span class="p">,</span> <span class="mf">1.0</span><span class="p">,</span> <span class="n">np</span><span class="o">.</span><span class="n">nan</span><span class="p">,</span> <span class="mf">2.0</span><span class="p">,</span> <span class="n">np</span><span class="o">.</span><span class="n">nan</span><span class="p">],</span> <span class="s2">"A"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="mi">4</span><span class="p">,</span> <span class="mi">5</span><span class="p">]})</span>

<span class="gp">In [215]: </span><span class="n">df</span>
<span class="gh">Out[215]: </span>
<span class="go">   key  A</span>
<span class="go">0  1.0  1</span>
<span class="go">1  1.0  2</span>
<span class="go">2  NaN  3</span>
<span class="go">3  2.0  4</span>
<span class="go">4  NaN  5</span>

<span class="gp">In [216]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"key"</span><span class="p">,</span> <span class="n">dropna</span><span class="o">=</span><span class="kc">True</span><span class="p">)</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[216]: </span>
<span class="go">     A</span>
<span class="go">key   </span>
<span class="go">1.0  3</span>
<span class="go">2.0  4</span>

<span class="gp">In [217]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"key"</span><span class="p">,</span> <span class="n">dropna</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[217]: </span>
<span class="go">     A</span>
<span class="go">key   </span>
<span class="go">1.0  3</span>
<span class="go">2.0  4</span>
<span class="go">NaN  8</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell85">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="grouping-with-ordered-factors">
<h3>Grouping with ordered factors<a class="headerlink" href="#grouping-with-ordered-factors" title="Link to this heading">#</a></h3>
<p>Categorical variables represented as instances of pandas’s <code class="docutils literal notranslate"><span class="pre">Categorical</span></code> class
can be used as group keys. If so, the order of the levels will be preserved. When
<code class="docutils literal notranslate"><span class="pre">observed=False</span></code> and <code class="docutils literal notranslate"><span class="pre">sort=False</span></code>, any unobserved categories will be at the
end of the result in order.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell86"><span></span><span class="gp">In [218]: </span><span class="n">days</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Categorical</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">values</span><span class="o">=</span><span class="p">[</span><span class="s2">"Wed"</span><span class="p">,</span> <span class="s2">"Mon"</span><span class="p">,</span> <span class="s2">"Thu"</span><span class="p">,</span> <span class="s2">"Mon"</span><span class="p">,</span> <span class="s2">"Wed"</span><span class="p">,</span> <span class="s2">"Sat"</span><span class="p">],</span>
<span class="gp">   .....: </span>    <span class="n">categories</span><span class="o">=</span><span class="p">[</span><span class="s2">"Mon"</span><span class="p">,</span> <span class="s2">"Tue"</span><span class="p">,</span> <span class="s2">"Wed"</span><span class="p">,</span> <span class="s2">"Thu"</span><span class="p">,</span> <span class="s2">"Fri"</span><span class="p">,</span> <span class="s2">"Sat"</span><span class="p">,</span> <span class="s2">"Sun"</span><span class="p">],</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [219]: </span><span class="n">data</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp">   .....: </span>   <span class="p">{</span>
<span class="gp">   .....: </span>       <span class="s2">"day"</span><span class="p">:</span> <span class="n">days</span><span class="p">,</span>
<span class="gp">   .....: </span>       <span class="s2">"workers"</span><span class="p">:</span> <span class="p">[</span><span class="mi">3</span><span class="p">,</span> <span class="mi">4</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">4</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span>
<span class="gp">   .....: </span>   <span class="p">}</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [220]: </span><span class="n">data</span>
<span class="gh">Out[220]: </span>
<span class="go">   day  workers</span>
<span class="go">0  Wed        3</span>
<span class="go">1  Mon        4</span>
<span class="go">2  Thu        1</span>
<span class="go">3  Mon        4</span>
<span class="go">4  Wed        2</span>
<span class="go">5  Sat        2</span>

<span class="gp">In [221]: </span><span class="n">data</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"day"</span><span class="p">,</span> <span class="n">observed</span><span class="o">=</span><span class="kc">False</span><span class="p">,</span> <span class="n">sort</span><span class="o">=</span><span class="kc">True</span><span class="p">)</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[221]: </span>
<span class="go">     workers</span>
<span class="go">day         </span>
<span class="go">Mon        8</span>
<span class="go">Tue        0</span>
<span class="go">Wed        5</span>
<span class="go">Thu        1</span>
<span class="go">Fri        0</span>
<span class="go">Sat        2</span>
<span class="go">Sun        0</span>

<span class="gp">In [222]: </span><span class="n">data</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"day"</span><span class="p">,</span> <span class="n">observed</span><span class="o">=</span><span class="kc">False</span><span class="p">,</span> <span class="n">sort</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[222]: </span>
<span class="go">     workers</span>
<span class="go">day         </span>
<span class="go">Wed        5</span>
<span class="go">Mon        8</span>
<span class="go">Thu        1</span>
<span class="go">Sat        2</span>
<span class="go">Tue        0</span>
<span class="go">Fri        0</span>
<span class="go">Sun        0</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell86">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="grouping-with-a-grouper-specification">
<span id="groupby-specify"></span><h3>Grouping with a grouper specification<a class="headerlink" href="#grouping-with-a-grouper-specification" title="Link to this heading">#</a></h3>
<p>You may need to specify a bit more data to properly group. You can
use the <code class="docutils literal notranslate"><span class="pre">pd.Grouper</span></code> to provide this local control.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell87"><span></span><span class="gp">In [223]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">datetime</span>

<span class="gp">In [224]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="p">{</span>
<span class="gp">   .....: </span>        <span class="s2">"Branch"</span><span class="p">:</span> <span class="s2">"A A A A A A A B"</span><span class="o">.</span><span class="n">split</span><span class="p">(),</span>
<span class="gp">   .....: </span>        <span class="s2">"Buyer"</span><span class="p">:</span> <span class="s2">"Carl Mark Carl Carl Joe Joe Joe Carl"</span><span class="o">.</span><span class="n">split</span><span class="p">(),</span>
<span class="gp">   .....: </span>        <span class="s2">"Quantity"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="mi">5</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">8</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">9</span><span class="p">,</span> <span class="mi">3</span><span class="p">],</span>
<span class="gp">   .....: </span>        <span class="s2">"Date"</span><span class="p">:</span> <span class="p">[</span>
<span class="gp">   .....: </span>            <span class="n">datetime</span><span class="o">.</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2013</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">13</span><span class="p">,</span> <span class="mi">0</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">datetime</span><span class="o">.</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2013</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">13</span><span class="p">,</span> <span class="mi">5</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">datetime</span><span class="o">.</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2013</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">20</span><span class="p">,</span> <span class="mi">0</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">datetime</span><span class="o">.</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2013</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="mi">0</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">datetime</span><span class="o">.</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2013</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">20</span><span class="p">,</span> <span class="mi">0</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">datetime</span><span class="o">.</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2013</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="mi">0</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">datetime</span><span class="o">.</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2013</span><span class="p">,</span> <span class="mi">12</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">12</span><span class="p">,</span> <span class="mi">0</span><span class="p">),</span>
<span class="gp">   .....: </span>            <span class="n">datetime</span><span class="o">.</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2013</span><span class="p">,</span> <span class="mi">12</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">14</span><span class="p">,</span> <span class="mi">0</span><span class="p">),</span>
<span class="gp">   .....: </span>        <span class="p">],</span>
<span class="gp">   .....: </span>    <span class="p">}</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [225]: </span><span class="n">df</span>
<span class="gh">Out[225]: </span>
<span class="go">  Branch Buyer  Quantity                Date</span>
<span class="go">0      A  Carl         1 2013-01-01 13:00:00</span>
<span class="go">1      A  Mark         3 2013-01-01 13:05:00</span>
<span class="go">2      A  Carl         5 2013-10-01 20:00:00</span>
<span class="go">3      A  Carl         1 2013-10-02 10:00:00</span>
<span class="go">4      A   Joe         8 2013-10-01 20:00:00</span>
<span class="go">5      A   Joe         1 2013-10-02 10:00:00</span>
<span class="go">6      A   Joe         9 2013-12-02 12:00:00</span>
<span class="go">7      B  Carl         3 2013-12-02 14:00:00</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell87">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>Groupby a specific column with the desired frequency. This is like resampling.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell88"><span></span><span class="gp">In [226]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">pd</span><span class="o">.</span><span class="n">Grouper</span><span class="p">(</span><span class="n">freq</span><span class="o">=</span><span class="s2">"1ME"</span><span class="p">,</span> <span class="n">key</span><span class="o">=</span><span class="s2">"Date"</span><span class="p">),</span> <span class="s2">"Buyer"</span><span class="p">])[[</span><span class="s2">"Quantity"</span><span class="p">]]</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[226]: </span>
<span class="go">                  Quantity</span>
<span class="go">Date       Buyer          </span>
<span class="go">2013-01-31 Carl          1</span>
<span class="go">           Mark          3</span>
<span class="go">2013-10-31 Carl          6</span>
<span class="go">           Joe           9</span>
<span class="go">2013-12-31 Carl          3</span>
<span class="go">           Joe           9</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell88">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>When <code class="docutils literal notranslate"><span class="pre">freq</span></code> is specified, the object returned by <code class="docutils literal notranslate"><span class="pre">pd.Grouper</span></code> will be an
instance of <code class="docutils literal notranslate"><span class="pre">pandas.api.typing.TimeGrouper</span></code>. When there is a column and index
with the same name, you can use <code class="docutils literal notranslate"><span class="pre">key</span></code> to group by the column and <code class="docutils literal notranslate"><span class="pre">level</span></code>
to group by the index.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell89"><span></span><span class="gp">In [227]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">set_index</span><span class="p">(</span><span class="s2">"Date"</span><span class="p">)</span>

<span class="gp">In [228]: </span><span class="n">df</span><span class="p">[</span><span class="s2">"Date"</span><span class="p">]</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">index</span> <span class="o">+</span> <span class="n">pd</span><span class="o">.</span><span class="n">offsets</span><span class="o">.</span><span class="n">MonthEnd</span><span class="p">(</span><span class="mi">2</span><span class="p">)</span>

<span class="gp">In [229]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">pd</span><span class="o">.</span><span class="n">Grouper</span><span class="p">(</span><span class="n">freq</span><span class="o">=</span><span class="s2">"6ME"</span><span class="p">,</span> <span class="n">key</span><span class="o">=</span><span class="s2">"Date"</span><span class="p">),</span> <span class="s2">"Buyer"</span><span class="p">])[[</span><span class="s2">"Quantity"</span><span class="p">]]</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[229]: </span>
<span class="go">                  Quantity</span>
<span class="go">Date       Buyer          </span>
<span class="go">2013-02-28 Carl          1</span>
<span class="go">           Mark          3</span>
<span class="go">2014-02-28 Carl          9</span>
<span class="go">           Joe          18</span>

<span class="gp">In [230]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">pd</span><span class="o">.</span><span class="n">Grouper</span><span class="p">(</span><span class="n">freq</span><span class="o">=</span><span class="s2">"6ME"</span><span class="p">,</span> <span class="n">level</span><span class="o">=</span><span class="s2">"Date"</span><span class="p">),</span> <span class="s2">"Buyer"</span><span class="p">])[[</span><span class="s2">"Quantity"</span><span class="p">]]</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span>
<span class="gh">Out[230]: </span>
<span class="go">                  Quantity</span>
<span class="go">Date       Buyer          </span>
<span class="go">2013-01-31 Carl          1</span>
<span class="go">           Mark          3</span>
<span class="go">2014-01-31 Carl          9</span>
<span class="go">           Joe          18</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell89">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="taking-the-first-rows-of-each-group">
<h3>Taking the first rows of each group<a class="headerlink" href="#taking-the-first-rows-of-each-group" title="Link to this heading">#</a></h3>
<p>Just like for a DataFrame or Series you can call head and tail on a groupby:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell90"><span></span><span class="gp">In [231]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">([[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">4</span><span class="p">],</span> <span class="p">[</span><span class="mi">5</span><span class="p">,</span> <span class="mi">6</span><span class="p">]],</span> <span class="n">columns</span><span class="o">=</span><span class="p">[</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span>

<span class="gp">In [232]: </span><span class="n">df</span>
<span class="gh">Out[232]: </span>
<span class="go">   A  B</span>
<span class="go">0  1  2</span>
<span class="go">1  1  4</span>
<span class="go">2  5  6</span>

<span class="gp">In [233]: </span><span class="n">g</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span>

<span class="gp">In [234]: </span><span class="n">g</span><span class="o">.</span><span class="n">head</span><span class="p">(</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[234]: </span>
<span class="go">   A  B</span>
<span class="go">0  1  2</span>
<span class="go">2  5  6</span>

<span class="gp">In [235]: </span><span class="n">g</span><span class="o">.</span><span class="n">tail</span><span class="p">(</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[235]: </span>
<span class="go">   A  B</span>
<span class="go">1  1  4</span>
<span class="go">2  5  6</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell90">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>This shows the first or last n rows from each group.</p>
</section>
<section id="taking-the-nth-row-of-each-group">
<span id="groupby-nth"></span><h3>Taking the nth row of each group<a class="headerlink" href="#taking-the-nth-row-of-each-group" title="Link to this heading">#</a></h3>
<p>To select the nth item from each group, use <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.nth.html#pandas.core.groupby.DataFrameGroupBy.nth" title="pandas.core.groupby.DataFrameGroupBy.nth"><code><span class="pre">DataFrameGroupBy.nth()</span></code></a> or
<a href="../reference/api/pandas.core.groupby.SeriesGroupBy.nth.html#pandas.core.groupby.SeriesGroupBy.nth" title="pandas.core.groupby.SeriesGroupBy.nth"><code><span class="pre">SeriesGroupBy.nth()</span></code></a>. Arguments supplied can be any integer, lists of integers,
slices, or lists of slices; see below for examples. When the nth element of a group
does not exist an error is <em>not</em> raised; instead no corresponding rows are returned.</p>
<p>In general this operation acts as a filtration. In certain cases it will also return
one row per group, making it also a reduction. However because in general it can
return zero or multiple rows per group, pandas treats it as a filtration in all cases.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell91"><span></span><span class="gp">In [236]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">([[</span><span class="mi">1</span><span class="p">,</span> <span class="n">np</span><span class="o">.</span><span class="n">nan</span><span class="p">],</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">4</span><span class="p">],</span> <span class="p">[</span><span class="mi">5</span><span class="p">,</span> <span class="mi">6</span><span class="p">]],</span> <span class="n">columns</span><span class="o">=</span><span class="p">[</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span>

<span class="gp">In [237]: </span><span class="n">g</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span>

<span class="gp">In [238]: </span><span class="n">g</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="mi">0</span><span class="p">)</span>
<span class="gh">Out[238]: </span>
<span class="go">   A    B</span>
<span class="go">0  1  NaN</span>
<span class="go">2  5  6.0</span>

<span class="gp">In [239]: </span><span class="n">g</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="o">-</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[239]: </span>
<span class="go">   A    B</span>
<span class="go">1  1  4.0</span>
<span class="go">2  5  6.0</span>

<span class="gp">In [240]: </span><span class="n">g</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[240]: </span>
<span class="go">   A    B</span>
<span class="go">1  1  4.0</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell91">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>If the nth element of a group does not exist, then no corresponding row is included
in the result. In particular, if the specified <code class="docutils literal notranslate"><span class="pre">n</span></code> is larger than any group, the
result will be an empty DataFrame.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell92"><span></span><span class="gp">In [241]: </span><span class="n">g</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="mi">5</span><span class="p">)</span>
<span class="gh">Out[241]: </span>
<span class="go">Empty DataFrame</span>
<span class="go">Columns: [A, B]</span>
<span class="go">Index: []</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell92">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>If you want to select the nth not-null item, use the <code class="docutils literal notranslate"><span class="pre">dropna</span></code> kwarg. For a DataFrame this should be either <code class="docutils literal notranslate"><span class="pre">'any'</span></code> or <code class="docutils literal notranslate"><span class="pre">'all'</span></code> just like you would pass to dropna:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell93"><span></span><span class="go"># nth(0) is the same as g.first()</span>
<span class="gp">In [242]: </span><span class="n">g</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="mi">0</span><span class="p">,</span> <span class="n">dropna</span><span class="o">=</span><span class="s2">"any"</span><span class="p">)</span>
<span class="gh">Out[242]: </span>
<span class="go">   A    B</span>
<span class="go">1  1  4.0</span>
<span class="go">2  5  6.0</span>

<span class="gp">In [243]: </span><span class="n">g</span><span class="o">.</span><span class="n">first</span><span class="p">()</span>
<span class="gh">Out[243]: </span>
<span class="go">     B</span>
<span class="go">A     </span>
<span class="go">1  4.0</span>
<span class="go">5  6.0</span>

<span class="go"># nth(-1) is the same as g.last()</span>
<span class="gp">In [244]: </span><span class="n">g</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="o">-</span><span class="mi">1</span><span class="p">,</span> <span class="n">dropna</span><span class="o">=</span><span class="s2">"any"</span><span class="p">)</span>
<span class="gh">Out[244]: </span>
<span class="go">   A    B</span>
<span class="go">1  1  4.0</span>
<span class="go">2  5  6.0</span>

<span class="gp">In [245]: </span><span class="n">g</span><span class="o">.</span><span class="n">last</span><span class="p">()</span>
<span class="gh">Out[245]: </span>
<span class="go">     B</span>
<span class="go">A     </span>
<span class="go">1  4.0</span>
<span class="go">5  6.0</span>

<span class="gp">In [246]: </span><span class="n">g</span><span class="o">.</span><span class="n">B</span><span class="o">.</span><span class="n">nth</span><span class="p">(</span><span class="mi">0</span><span class="p">,</span> <span class="n">dropna</span><span class="o">=</span><span class="s2">"all"</span><span class="p">)</span>
<span class="gh">Out[246]: </span>
<span class="go">1    4.0</span>
<span class="go">2    6.0</span>
<span class="go">Name: B, dtype: float64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell93">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>You can also select multiple rows from each group by specifying multiple nth values as a list of ints.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell94"><span></span><span class="gp">In [247]: </span><span class="n">business_dates</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">date_range</span><span class="p">(</span><span class="n">start</span><span class="o">=</span><span class="s2">"4/1/2014"</span><span class="p">,</span> <span class="n">end</span><span class="o">=</span><span class="s2">"6/30/2014"</span><span class="p">,</span> <span class="n">freq</span><span class="o">=</span><span class="s2">"B"</span><span class="p">)</span>

<span class="gp">In [248]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="mi">1</span><span class="p">,</span> <span class="n">index</span><span class="o">=</span><span class="n">business_dates</span><span class="p">,</span> <span class="n">columns</span><span class="o">=</span><span class="p">[</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"b"</span><span class="p">])</span>

<span class="go"># get the first, 4th, and last date index for each month</span>
<span class="gp">In [249]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">year</span><span class="p">,</span> <span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">month</span><span class="p">])</span><span class="o">.</span><span class="n">nth</span><span class="p">([</span><span class="mi">0</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="o">-</span><span class="mi">1</span><span class="p">])</span>
<span class="gh">Out[249]: </span>
<span class="go">            a  b</span>
<span class="go">2014-04-01  1  1</span>
<span class="go">2014-04-04  1  1</span>
<span class="go">2014-04-30  1  1</span>
<span class="go">2014-05-01  1  1</span>
<span class="go">2014-05-06  1  1</span>
<span class="go">2014-05-30  1  1</span>
<span class="go">2014-06-02  1  1</span>
<span class="go">2014-06-05  1  1</span>
<span class="go">2014-06-30  1  1</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell94">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>You may also use slices or lists of slices.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell95"><span></span><span class="gp">In [250]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">year</span><span class="p">,</span> <span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">month</span><span class="p">])</span><span class="o">.</span><span class="n">nth</span><span class="p">[</span><span class="mi">1</span><span class="p">:]</span>
<span class="gh">Out[250]: </span>
<span class="go">            a  b</span>
<span class="go">2014-04-02  1  1</span>
<span class="go">2014-04-03  1  1</span>
<span class="go">2014-04-04  1  1</span>
<span class="go">2014-04-07  1  1</span>
<span class="go">2014-04-08  1  1</span>
<span class="go">...        .. ..</span>
<span class="go">2014-06-24  1  1</span>
<span class="go">2014-06-25  1  1</span>
<span class="go">2014-06-26  1  1</span>
<span class="go">2014-06-27  1  1</span>
<span class="go">2014-06-30  1  1</span>

<span class="go">[62 rows x 2 columns]</span>

<span class="gp">In [251]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">year</span><span class="p">,</span> <span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">month</span><span class="p">])</span><span class="o">.</span><span class="n">nth</span><span class="p">[</span><span class="mi">1</span><span class="p">:,</span> <span class="p">:</span><span class="o">-</span><span class="mi">1</span><span class="p">]</span>
<span class="gh">Out[251]: </span>
<span class="go">            a  b</span>
<span class="go">2014-04-01  1  1</span>
<span class="go">2014-04-02  1  1</span>
<span class="go">2014-04-03  1  1</span>
<span class="go">2014-04-04  1  1</span>
<span class="go">2014-04-07  1  1</span>
<span class="go">...        .. ..</span>
<span class="go">2014-06-24  1  1</span>
<span class="go">2014-06-25  1  1</span>
<span class="go">2014-06-26  1  1</span>
<span class="go">2014-06-27  1  1</span>
<span class="go">2014-06-30  1  1</span>

<span class="go">[65 rows x 2 columns]</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell95">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="enumerate-group-items">
<h3>Enumerate group items<a class="headerlink" href="#enumerate-group-items" title="Link to this heading">#</a></h3>
<p>To see the order in which each row appears within its group, use the
<code class="docutils literal notranslate"><span class="pre">cumcount</span></code> method:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell96"><span></span><span class="gp">In [252]: </span><span class="n">dfg</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="nb">list</span><span class="p">(</span><span class="s2">"aaabba"</span><span class="p">),</span> <span class="n">columns</span><span class="o">=</span><span class="p">[</span><span class="s2">"A"</span><span class="p">])</span>

<span class="gp">In [253]: </span><span class="n">dfg</span>
<span class="gh">Out[253]: </span>
<span class="go">   A</span>
<span class="go">0  a</span>
<span class="go">1  a</span>
<span class="go">2  a</span>
<span class="go">3  b</span>
<span class="go">4  b</span>
<span class="go">5  a</span>

<span class="gp">In [254]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">cumcount</span><span class="p">()</span>
<span class="gh">Out[254]: </span>
<span class="go">0    0</span>
<span class="go">1    1</span>
<span class="go">2    2</span>
<span class="go">3    0</span>
<span class="go">4    1</span>
<span class="go">5    3</span>
<span class="go">dtype: int64</span>

<span class="gp">In [255]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">cumcount</span><span class="p">(</span><span class="n">ascending</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="gh">Out[255]: </span>
<span class="go">0    3</span>
<span class="go">1    2</span>
<span class="go">2    1</span>
<span class="go">3    1</span>
<span class="go">4    0</span>
<span class="go">5    0</span>
<span class="go">dtype: int64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell96">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="enumerate-groups">
<span id="groupby-ngroup"></span><h3>Enumerate groups<a class="headerlink" href="#enumerate-groups" title="Link to this heading">#</a></h3>
<p>To see the ordering of the groups (as opposed to the order of rows
within a group given by <code class="docutils literal notranslate"><span class="pre">cumcount</span></code>) you can use
<a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.ngroup.html#pandas.core.groupby.DataFrameGroupBy.ngroup" title="pandas.core.groupby.DataFrameGroupBy.ngroup"><code><span class="pre">DataFrameGroupBy.ngroup()</span></code></a>.</p>
<p>Note that the numbers given to the groups match the order in which the
groups would be seen when iterating over the groupby object, not the
order they are first observed.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell97"><span></span><span class="gp">In [256]: </span><span class="n">dfg</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="nb">list</span><span class="p">(</span><span class="s2">"aaabba"</span><span class="p">),</span> <span class="n">columns</span><span class="o">=</span><span class="p">[</span><span class="s2">"A"</span><span class="p">])</span>

<span class="gp">In [257]: </span><span class="n">dfg</span>
<span class="gh">Out[257]: </span>
<span class="go">   A</span>
<span class="go">0  a</span>
<span class="go">1  a</span>
<span class="go">2  a</span>
<span class="go">3  b</span>
<span class="go">4  b</span>
<span class="go">5  a</span>

<span class="gp">In [258]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">ngroup</span><span class="p">()</span>
<span class="gh">Out[258]: </span>
<span class="go">0    0</span>
<span class="go">1    0</span>
<span class="go">2    0</span>
<span class="go">3    1</span>
<span class="go">4    1</span>
<span class="go">5    0</span>
<span class="go">dtype: int64</span>

<span class="gp">In [259]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"A"</span><span class="p">)</span><span class="o">.</span><span class="n">ngroup</span><span class="p">(</span><span class="n">ascending</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="gh">Out[259]: </span>
<span class="go">0    1</span>
<span class="go">1    1</span>
<span class="go">2    1</span>
<span class="go">3    0</span>
<span class="go">4    0</span>
<span class="go">5    1</span>
<span class="go">dtype: int64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell97">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="plotting">
<h3>Plotting<a class="headerlink" href="#plotting" title="Link to this heading">#</a></h3>
<p>Groupby also works with some plotting methods.  In this case, suppose we
suspect that the values in column 1 are 3 times higher on average in group “B”.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell98"><span></span><span class="gp">In [260]: </span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">seed</span><span class="p">(</span><span class="mi">1234</span><span class="p">)</span>

<span class="gp">In [261]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">50</span><span class="p">,</span> <span class="mi">2</span><span class="p">))</span>

<span class="gp">In [262]: </span><span class="n">df</span><span class="p">[</span><span class="s2">"g"</span><span class="p">]</span> <span class="o">=</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">choice</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">],</span> <span class="n">size</span><span class="o">=</span><span class="mi">50</span><span class="p">)</span>

<span class="gp">In [263]: </span><span class="n">df</span><span class="o">.</span><span class="n">loc</span><span class="p">[</span><span class="n">df</span><span class="p">[</span><span class="s2">"g"</span><span class="p">]</span> <span class="o">==</span> <span class="s2">"B"</span><span class="p">,</span> <span class="mi">1</span><span class="p">]</span> <span class="o">+=</span> <span class="mi">3</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell98">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>We can easily visualize this with a boxplot:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell99"><span></span><span class="gp">In [264]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"g"</span><span class="p">)</span><span class="o">.</span><span class="n">boxplot</span><span class="p">()</span>
<span class="gh">Out[264]: </span>
<span class="go">A         Axes(0.1,0.15;0.363636x0.75)</span>
<span class="go">B    Axes(0.536364,0.15;0.363636x0.75)</span>
<span class="go">dtype: object</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell99">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<img alt="../_images/groupby_boxplot.png" src="../_images/groupby_boxplot.png">
<p>The result of calling <code class="docutils literal notranslate"><span class="pre">boxplot</span></code> is a dictionary whose keys are the values
of our grouping column <code class="docutils literal notranslate"><span class="pre">g</span></code> (“A” and “B”). The values of the resulting dictionary
can be controlled by the <code class="docutils literal notranslate"><span class="pre">return_type</span></code> keyword of <code class="docutils literal notranslate"><span class="pre">boxplot</span></code>.
See the <a href="visualization.html#visualization-box"><span class="std std-ref">visualization documentation</span></a> for more.</p>
<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p>For historical reasons, <code class="docutils literal notranslate"><span class="pre">df.groupby("g").boxplot()</span></code> is not equivalent
to <code class="docutils literal notranslate"><span class="pre">df.boxplot(by="g")</span></code>. See <a href="visualization.html#visualization-box-return"><span class="std std-ref">here</span></a> for
an explanation.</p>
</div>
</section>
<section id="piping-function-calls">
<span id="groupby-pipe"></span><h3>Piping function calls<a class="headerlink" href="#piping-function-calls" title="Link to this heading">#</a></h3>
<p>Similar to the functionality provided by <code class="docutils literal notranslate"><span class="pre">DataFrame</span></code> and <code class="docutils literal notranslate"><span class="pre">Series</span></code>, functions
that take <code class="docutils literal notranslate"><span class="pre">GroupBy</span></code> objects can be chained together using a <code class="docutils literal notranslate"><span class="pre">pipe</span></code> method to
allow for a cleaner, more readable syntax. To read about <code class="docutils literal notranslate"><span class="pre">.pipe</span></code> in general terms,
see <a href="basics.html#basics-pipe"><span class="std std-ref">here</span></a>.</p>
<p>Combining <code class="docutils literal notranslate"><span class="pre">.groupby</span></code> and <code class="docutils literal notranslate"><span class="pre">.pipe</span></code> is often useful when you need to reuse
GroupBy objects.</p>
<p>As an example, imagine having a DataFrame with columns for stores, products,
revenue and quantity sold. We’d like to do a groupwise calculation of <em>prices</em>
(i.e. revenue/quantity) per store and per product. We could do this in a
multi-step operation, but expressing it in terms of piping can make the
code more readable. First we set the data:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell100"><span></span><span class="gp">In [265]: </span><span class="n">n</span> <span class="o">=</span> <span class="mi">1000</span>

<span class="gp">In [266]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="p">{</span>
<span class="gp">   .....: </span>        <span class="s2">"Store"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">choice</span><span class="p">([</span><span class="s2">"Store_1"</span><span class="p">,</span> <span class="s2">"Store_2"</span><span class="p">],</span> <span class="n">n</span><span class="p">),</span>
<span class="gp">   .....: </span>        <span class="s2">"Product"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">choice</span><span class="p">([</span><span class="s2">"Product_1"</span><span class="p">,</span> <span class="s2">"Product_2"</span><span class="p">],</span> <span class="n">n</span><span class="p">),</span>
<span class="gp">   .....: </span>        <span class="s2">"Revenue"</span><span class="p">:</span> <span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">random</span><span class="p">(</span><span class="n">n</span><span class="p">)</span> <span class="o">*</span> <span class="mi">50</span> <span class="o">+</span> <span class="mi">10</span><span class="p">)</span><span class="o">.</span><span class="n">round</span><span class="p">(</span><span class="mi">2</span><span class="p">),</span>
<span class="gp">   .....: </span>        <span class="s2">"Quantity"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randint</span><span class="p">(</span><span class="mi">1</span><span class="p">,</span> <span class="mi">10</span><span class="p">,</span> <span class="n">size</span><span class="o">=</span><span class="n">n</span><span class="p">),</span>
<span class="gp">   .....: </span>    <span class="p">}</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [267]: </span><span class="n">df</span><span class="o">.</span><span class="n">head</span><span class="p">(</span><span class="mi">2</span><span class="p">)</span>
<span class="gh">Out[267]: </span>
<span class="go">     Store    Product  Revenue  Quantity</span>
<span class="go">0  Store_2  Product_1    26.12         1</span>
<span class="go">1  Store_2  Product_1    28.86         1</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell100">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>We now find the prices per store/product.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell101"><span></span><span class="gp">In [268]: </span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"Store"</span><span class="p">,</span> <span class="s2">"Product"</span><span class="p">])</span>
<span class="gp">   .....: </span>    <span class="o">.</span><span class="n">pipe</span><span class="p">(</span><span class="k">lambda</span> <span class="n">grp</span><span class="p">:</span> <span class="n">grp</span><span class="o">.</span><span class="n">Revenue</span><span class="o">.</span><span class="n">sum</span><span class="p">()</span> <span class="o">/</span> <span class="n">grp</span><span class="o">.</span><span class="n">Quantity</span><span class="o">.</span><span class="n">sum</span><span class="p">())</span>
<span class="gp">   .....: </span>    <span class="o">.</span><span class="n">unstack</span><span class="p">()</span>
<span class="gp">   .....: </span>    <span class="o">.</span><span class="n">round</span><span class="p">(</span><span class="mi">2</span><span class="p">)</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>
<span class="gh">Out[268]: </span>
<span class="go">Product  Product_1  Product_2</span>
<span class="go">Store                        </span>
<span class="go">Store_1       6.82       7.05</span>
<span class="go">Store_2       6.30       6.64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell101">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>Piping can also be expressive when you want to deliver a grouped object to some
arbitrary function, for example:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell102"><span></span><span class="gp">In [269]: </span><span class="k">def</span><span class="w"> </span><span class="nf">mean</span><span class="p">(</span><span class="n">groupby</span><span class="p">):</span>
<span class="gp">   .....: </span>    <span class="k">return</span> <span class="n">groupby</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>
<span class="gp">   .....: </span>

<span class="gp">In [270]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"Store"</span><span class="p">,</span> <span class="s2">"Product"</span><span class="p">])</span><span class="o">.</span><span class="n">pipe</span><span class="p">(</span><span class="n">mean</span><span class="p">)</span>
<span class="gh">Out[270]: </span>
<span class="go">                     Revenue  Quantity</span>
<span class="go">Store   Product                       </span>
<span class="go">Store_1 Product_1  34.622727  5.075758</span>
<span class="go">        Product_2  35.482815  5.029630</span>
<span class="go">Store_2 Product_1  32.972837  5.237589</span>
<span class="go">        Product_2  34.684360  5.224000</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell102">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
<p>Here <code class="docutils literal notranslate"><span class="pre">mean</span></code> takes a GroupBy object and finds the mean of the Revenue and Quantity
columns respectively for each Store-Product combination. The <code class="docutils literal notranslate"><span class="pre">mean</span></code> function can
be any function that takes in a GroupBy object; the <code class="docutils literal notranslate"><span class="pre">.pipe</span></code> will pass the GroupBy
object as a parameter into the function you specify.</p>
</section>
</section>
<section id="examples">
<h2>Examples<a class="headerlink" href="#examples" title="Link to this heading">#</a></h2>
<section id="multi-column-factorization">
<span id="groupby-multicolumn-factorization"></span><h3>Multi-column factorization<a class="headerlink" href="#multi-column-factorization" title="Link to this heading">#</a></h3>
<p>By using <a href="../reference/api/pandas.core.groupby.DataFrameGroupBy.ngroup.html#pandas.core.groupby.DataFrameGroupBy.ngroup" title="pandas.core.groupby.DataFrameGroupBy.ngroup"><code><span class="pre">DataFrameGroupBy.ngroup()</span></code></a>, we can extract
information about the groups in a way similar to <a href="../reference/api/pandas.factorize.html#pandas.factorize" title="pandas.factorize"><code class="xref py py-func docutils literal notranslate"><span class="pre">factorize()</span></code></a> (as described
further in the <a href="reshaping.html#reshaping-factorize"><span class="std std-ref">reshaping API</span></a>) but which applies
naturally to multiple columns of mixed type and different
sources. This can be useful as an intermediate categorical-like step
in processing, when the relationships between the group rows are more
important than their content, or as input to an algorithm which only
accepts the integer encoding. (For more information about support in
pandas for full categorical data, see the <a href="categorical.html#categorical"><span class="std std-ref">Categorical
introduction</span></a> and the
<a href="../reference/arrays.html#api-arrays-categorical"><span class="std std-ref">API documentation</span></a>.)</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell103"><span></span><span class="gp">In [271]: </span><span class="n">dfg</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">({</span><span class="s2">"A"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">3</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span> <span class="s2">"B"</span><span class="p">:</span> <span class="nb">list</span><span class="p">(</span><span class="s2">"aaaba"</span><span class="p">)})</span>

<span class="gp">In [272]: </span><span class="n">dfg</span>
<span class="gh">Out[272]: </span>
<span class="go">   A  B</span>
<span class="go">0  1  a</span>
<span class="go">1  1  a</span>
<span class="go">2  2  a</span>
<span class="go">3  3  b</span>
<span class="go">4  2  a</span>

<span class="gp">In [273]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="s2">"B"</span><span class="p">])</span><span class="o">.</span><span class="n">ngroup</span><span class="p">()</span>
<span class="gh">Out[273]: </span>
<span class="go">0    0</span>
<span class="go">1    0</span>
<span class="go">2    1</span>
<span class="go">3    2</span>
<span class="go">4    1</span>
<span class="go">dtype: int64</span>

<span class="gp">In [274]: </span><span class="n">dfg</span><span class="o">.</span><span class="n">groupby</span><span class="p">([</span><span class="s2">"A"</span><span class="p">,</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">]])</span><span class="o">.</span><span class="n">ngroup</span><span class="p">()</span>
<span class="gh">Out[274]: </span>
<span class="go">0    0</span>
<span class="go">1    0</span>
<span class="go">2    1</span>
<span class="go">3    3</span>
<span class="go">4    2</span>
<span class="go">dtype: int64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell103">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="groupby-by-indexer-to-resample-data">
<h3>Groupby by indexer to ‘resample’ data<a class="headerlink" href="#groupby-by-indexer-to-resample-data" title="Link to this heading">#</a></h3>
<p>Resampling produces new hypothetical samples (resamples) from already existing observed data or from a model that generates data. These new samples are similar to the pre-existing samples.</p>
<p>In order for resample to work on indices that are non-datetimelike, the following procedure can be utilized.</p>
<p>In the following examples, <strong>df.index // 5</strong> returns an integer array which is used to determine what gets selected for the groupby operation.</p>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>The example below shows how we can downsample by consolidation of samples into fewer ones.
Here by using <strong>df.index // 5</strong>, we are aggregating the samples in bins. By applying <strong>std()</strong>
function, we aggregate the information contained in many samples into a small subset of values
which is their standard deviation thereby reducing the number of samples.</p>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell104"><span></span><span class="gp">In [275]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">10</span><span class="p">,</span> <span class="mi">2</span><span class="p">))</span>

<span class="gp">In [276]: </span><span class="n">df</span>
<span class="gh">Out[276]: </span>
<span class="go">          0         1</span>
<span class="go">0 -0.793893  0.321153</span>
<span class="go">1  0.342250  1.618906</span>
<span class="go">2 -0.975807  1.918201</span>
<span class="go">3 -0.810847 -1.405919</span>
<span class="go">4 -1.977759  0.461659</span>
<span class="go">5  0.730057 -1.316938</span>
<span class="go">6 -0.751328  0.528290</span>
<span class="go">7 -0.257759 -1.081009</span>
<span class="go">8  0.505895 -1.701948</span>
<span class="go">9 -1.006349  0.020208</span>

<span class="gp">In [277]: </span><span class="n">df</span><span class="o">.</span><span class="n">index</span> <span class="o">//</span> <span class="mi">5</span>
<span class="gh">Out[277]: </span><span class="go">Index([0, 0, 0, 0, 0, 1, 1, 1, 1, 1], dtype='int64')</span>

<span class="gp">In [278]: </span><span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">df</span><span class="o">.</span><span class="n">index</span> <span class="o">//</span> <span class="mi">5</span><span class="p">)</span><span class="o">.</span><span class="n">std</span><span class="p">()</span>
<span class="gh">Out[278]: </span>
<span class="go">          0         1</span>
<span class="go">0  0.823647  1.312912</span>
<span class="go">1  0.760109  0.942941</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell104">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
<section id="returning-a-series-to-propagate-names">
<h3>Returning a Series to propagate names<a class="headerlink" href="#returning-a-series-to-propagate-names" title="Link to this heading">#</a></h3>
<p>Group DataFrame columns, compute a set of metrics and return a named Series.
The Series name is used as the name for the column index. This is especially
useful in conjunction with reshaping operations such as stacking, in which the
column index name will be used as the name of the inserted column:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell105"><span></span><span class="gp">In [279]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="p">{</span>
<span class="gp">   .....: </span>        <span class="s2">"a"</span><span class="p">:</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span>
<span class="gp">   .....: </span>        <span class="s2">"b"</span><span class="p">:</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">],</span>
<span class="gp">   .....: </span>        <span class="s2">"c"</span><span class="p">:</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">],</span>
<span class="gp">   .....: </span>        <span class="s2">"d"</span><span class="p">:</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">],</span>
<span class="gp">   .....: </span>    <span class="p">}</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [280]: </span><span class="k">def</span><span class="w"> </span><span class="nf">compute_metrics</span><span class="p">(</span><span class="n">x</span><span class="p">):</span>
<span class="gp">   .....: </span>    <span class="n">result</span> <span class="o">=</span> <span class="p">{</span><span class="s2">"b_sum"</span><span class="p">:</span> <span class="n">x</span><span class="p">[</span><span class="s2">"b"</span><span class="p">]</span><span class="o">.</span><span class="n">sum</span><span class="p">(),</span> <span class="s2">"c_mean"</span><span class="p">:</span> <span class="n">x</span><span class="p">[</span><span class="s2">"c"</span><span class="p">]</span><span class="o">.</span><span class="n">mean</span><span class="p">()}</span>
<span class="gp">   .....: </span>    <span class="k">return</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">result</span><span class="p">,</span> <span class="n">name</span><span class="o">=</span><span class="s2">"metrics"</span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [281]: </span><span class="n">result</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="s2">"a"</span><span class="p">)</span><span class="o">.</span><span class="n">apply</span><span class="p">(</span><span class="n">compute_metrics</span><span class="p">,</span> <span class="n">include_groups</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>

<span class="gp">In [282]: </span><span class="n">result</span>
<span class="gh">Out[282]: </span>
<span class="go">metrics  b_sum  c_mean</span>
<span class="go">a                     </span>
<span class="go">0          2.0     0.5</span>
<span class="go">1          2.0     0.5</span>
<span class="go">2          2.0     0.5</span>

<span class="gp">In [283]: </span><span class="n">result</span><span class="o">.</span><span class="n">stack</span><span class="p">(</span><span class="n">future_stack</span><span class="o">=</span><span class="kc">True</span><span class="p">)</span>
<span class="gh">Out[283]: </span>
<span class="go">a  metrics</span>
<span class="go">0  b_sum      2.0</span>
<span class="go">   c_mean     0.5</span>
<span class="go">1  b_sum      2.0</span>
<span class="go">   c_mean     0.5</span>
<span class="go">2  b_sum      2.0</span>
<span class="go">   c_mean     0.5</span>
<span class="go">dtype: float64</span>
</pre><button class="copybtn o-tooltip--left" data-tooltip="Copy" data-clipboard-target="#codecell105">
      <svg xmlns="http://www.w3.org/2000/svg" class="icon icon-tabler icon-tabler-copy" width="44" height="44" viewBox="0 0 24 24" stroke-width="1.5" stroke="#000000" fill="none" stroke-linecap="round" stroke-linejoin="round">
  <title>Copy to clipboard</title>
  <path stroke="none" d="M0 0h24v24H0z" fill="none"></path>
  <rect x="8" y="8" width="12" height="12" rx="2"></rect>
  <path d="M16 8v-2a2 2 0 0 0 -2 -2h-8a2 2 0 0 0 -2 2v8a2 2 0 0 0 2 2h2"></path>
</svg>
    </button></div>
</div>
</section>
</section>
</section>